# 01, Data Understanding

**O projeto.** Um agente de IA que apoia o analista de crédito de um banco na análise manual de pedidos de financiamento imobiliário do FHA. Para cada pedido, o agente recomenda aprovar, aprovar com fatores compensatórios, negar ou suspender pedindo documentos, e cita a seção da norma oficial que sustenta a recomendação. O analista decide. A recomendação do agente é medida contra a decisão real dos bancos, publicada em bases públicas.

**O que este notebook faz.** Antes de construir o agente, confere se os dados públicos sustentam essa comparação:

1. Checagem 1: nas negativas do HMDA, o motivo aparece nos números do pedido?
2. Checagem 2: a NSMO traz DTI e score exatos e a reserva do cliente, que o HMDA não publica?
3. Resultado das checagens.

Depois, explora as duas bases e a norma, com as ausências, o texto oficial dos limites, campos que a norma cita e uma prévia de como a norma se aplica ao dado.

**Siglas usadas ao longo do notebook.**

| Sigla | Significado | O que é no projeto |
|---|---|---|
| FHA | Federal Housing Administration | órgão do governo americano que garante financiamentos imobiliários de entrada baixa; o projeto trata só desses pedidos |
| HUD | Department of Housing and Urban Development | ministério da habitação dos EUA, dono do FHA e autor da norma (Handbook 4000.1) |
| HMDA | Home Mortgage Disclosure Act | lei que obriga bancos e financeiras acima de um volume mínimo de operações a publicar seus pedidos de financiamento imobiliário (os menores ficam dispensados); dá nome à base pública de pedidos |
| CFPB | Consumer Financial Protection Bureau | agência de defesa do consumidor financeiro que publica o HMDA |
| NSMO | National Survey of Mortgage Originations | pesquisa com quem pegou financiamento; traz score e reserva, mas só de aprovados |
| FHFA | Federal Housing Finance Agency | agência que publica a NSMO junto com o CFPB |
| DTI | Debt to Income | soma das dívidas mensais (com a nova prestação) dividida pela renda mensal |
| PTI | Payment to Income | só a prestação do imóvel dividida pela renda mensal |
| LTV | Loan to Value | valor financiado dividido pelo valor do imóvel |
| CLTV | Combined Loan to Value | como o LTV, somando todas as dívidas garantidas pelo imóvel |
| AUS | Automated Underwriting System | sistema automático de aprovação; no FHA, o TOTAL (Technology Open To Approved Lenders) Scorecard |
| VA | Department of Veterans Affairs | órgão dos veteranos, cuja tabela de renda residual o Handbook adota |

**O que a norma diz, em resumo.** Quando o sistema automático do FHA não aprova sozinho, o pedido vai para um analista, que segue o HUD Handbook 4000.1. Os limites que este notebook usa:

| DTI do pedido | O que a norma exige na análise manual |
|---|---|
| até 43% | nada além do básico (score a partir de 500) |
| 44% a 47% | score a partir de 580 e um fator compensatório |
| 48% a 50% | score a partir de 580 e dois fatores compensatórios |
| acima de 50% | a análise manual não aprova |

Cada faixa tem também um teto para o PTI, a prestação da casa sobre a renda: 31%, 37% e 40%.

**O que é fator compensatório.** Acima de 43% de DTI, a norma ainda permite aprovar se o cliente tiver algo que reduza o risco de calote. Esse algo é o fator compensatório. Exemplo: um cliente com DTI de 46% que tem guardado o equivalente a três prestações continua pagando mesmo se a renda apertar por alguns meses; a reserva compensa o DTI alto. A norma lista quatro fatores, que são reserva guardada, prestação nova pouco maior que o gasto atual com moradia, renda residual (o que sobra depois de pagar as contas) e renda extra que não entrou no cálculo.

**Quanto do imóvel pode ser financiado.** A dívida também tem teto, medido pelo CLTV, e o teto depende da finalidade do pedido. Exemplo com um imóvel de US$ 300 mil:

| Finalidade | O que o cliente faz | Teto | Quanto pode pegar | Seção |
|---|---|---|---|---|
| Compra | compra o imóvel e dá uma entrada | 96,5% (score a partir de 580) | US$ 289,5 mil | II.A.2 |
| Refinanciamento com saque | troca a dívida antiga por uma nova e leva dinheiro | 80% | US$ 240 mil | II.A.8 |
| Refinanciamento sem saque | troca a dívida antiga por uma nova, sem levar dinheiro | 97,75% | US$ 293,25 mil | II.A.8 |

Três detalhes da norma mudam a leitura do dado:

- **Seguro financiado.** O FHA cobra no início um seguro de 1,75% do empréstimo, que o cliente pode somar à dívida em vez de pagar à vista. O teto vale antes desse seguro (II.A.2), então numa compra a dívida chega a 98,2% do imóvel sem ferir a norma.
- **Segundo financiamento de órgão público.** Programas de governo podem emprestar ao cliente o dinheiro da entrada, num segundo financiamento sobre o mesmo imóvel. Nesse caso a norma não impõe teto ao CLTV (II.A.5, Secondary Financing), e a dívida somada pode passar do valor do imóvel.
- **Três tipos de refinanciamento sem saque.** Dois têm teto de 97,75%. O terceiro, o *Streamline*, refinanciamento simplificado de quem já tem financiamento FHA, dispensa a avaliação do imóvel e não tem teto de CLTV, porque a dívida nova fica limitada ao saldo da antiga (II.A.8).

O texto oficial dos limites de DTI está na seção 5.

Fontes: HMDA público de 2025 (CFPB), EUA inteiro, só pedidos FHA, baixado em 29 de setembro de 2026 pela API do HMDA Data Browser (https://ffiec.cfpb.gov/v2/data-browser-api/view/nationwide/csv). NSMO Public Use File, versão 70.

## 0. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.hmda import (
    ACTION_TAKEN,
    DENIAL_INVISIBLE,
    DENIAL_REASON,
    DENIAL_VISIBLE,
    DTI_ORDER,
    HMDA_CSV,
    LOAN_PURPOSE,
    dti_above,
    dti_band,
    load_hmda,
)

SEED = 42
pd.set_option('display.max_columns', 30)

In [2]:
hmda = load_hmda()
print(HMDA_CSV.name, hmda.shape)

hmda_2025_fha.csv (2062934, 19)


## 1. Checagem 1: a negativa do HMDA aparece nos campos públicos?

**A pergunta.** O agente vai receber os números de um pedido, como renda, valor do imóvel e DTI, e recomendar aprovar ou negar. Para comparar essa recomendação com o que o banco fez de verdade, o motivo da negativa precisa estar, de algum jeito, nos números publicados no HMDA. Exemplo: se o banco negou um pedido por DTI, espera-se ver nesse pedido um DTI alto. Se os negados por DTI tiverem DTI parecido com o dos aprovados, o agente não tem como acertar a decisão do banco lendo só o HMDA, e o teste principal perde sentido.

**O caminho da seção.**

- 1.0: como o HMDA é organizado e de onde vêm os significados dos códigos.
- 1.1: como os pedidos terminaram.
- 1.2: dez pedidos negados de verdade, para ver o dado cru.
- 1.3: por qual motivo os bancos negam.
- 1.4: quem foi negado por DTI tem mesmo DTI alto?
- 1.5: quem foi negado por garantia tem mesmo valor financiado alto?
- 1.6: existe alguma outra combinação de campos que denuncie a negativa?
- 1.7: resultado da checagem.

**Duas regras para responder.** Os cortes abaixo foram definidos antes de olhar os dados. Cada regra ganha um apelido, usado no resto da seção.

1. **Regra dos 25%:** pelo menos 25% das negativas têm como motivo principal DTI ou garantia, os dois motivos que têm um campo correspondente no HMDA (o DTI e o CLTV). Em outras palavras, de cada 4 pedidos negados, pelo menos 1 precisa ter sido negado por um motivo que aparece nos números. Abaixo disso, o agente seria testado só num canto pequeno das negativas.
2. **Regra do dobro:** entre os negados por DTI, a parcela com DTI acima de 43% é pelo menos o dobro da parcela entre os originados, isto é, os pedidos em que o financiamento foi concedido (seção 1.1). 43% é o limite da norma para aprovar sem nenhuma condição extra. Exemplo com números redondos: se 30 de cada 100 aprovados têm DTI acima de 43%, pelo menos 60 de cada 100 negados por DTI precisam ter. Se der 35 contra 30, o DTI alto aparece quase igual nos dois grupos, não ajuda a separar quem foi negado, e o agente decidiria no escuro.

Se as duas regras valerem, o HMDA serve como está para medir o agente. Se alguma falhar, a forma de medir precisa mudar.

**Como ler as próximas tabelas.** Cada tabela vem com uma pergunta antes e, depois, três itens: onde olhar, o que se vê e o que significa.

### 1.0 Como ler o HMDA

**Uma linha é um pedido de financiamento** feito a um banco em 2025. O arquivo baixado já vem filtrado para pedidos FHA. São 2.062.934 linhas, e o notebook carrega só as 19 colunas que interessam ao projeto.

**Os campos vêm como códigos, não como texto.** O CFPB publica a decisão do banco como `1`, `2`, `3` e assim por diante. O significado de cada código está no dicionário oficial da base, *Public HMDA, LAR Data Fields* (https://ffiec.cfpb.gov/documentation/publications/loan-level-datasets/lar-data-fields). Os rótulos em português usados aqui são tradução direta desse dicionário, guardados em `src/hmda.py` (`ACTION_TAKEN`, `DENIAL_REASON`, `LOAN_PURPOSE`, `OCCUPANCY`, `LIEN`) para que todo notebook use a mesma tradução. A célula abaixo mostra as duas tabelas de códigos mais usadas nesta seção.

**As 19 colunas carregadas.**

| Coluna | O que é | Onde aparece |
|---|---|---|
| `action_taken` | como o pedido terminou (8 códigos) | em todo o notebook |
| `denial_reason-1` a `denial_reason-4` | o motivo da negativa; o banco pode informar até quatro, e o `-1` é o principal | 1.2, 1.3 e daí em diante |
| `debt_to_income_ratio` | o DTI do pedido | 1.2, 1.4, 1.6 |
| `loan_to_value_ratio` | apesar do nome, o CLTV do pedido | 1.2, 1.5, 1.6 |
| `loan_purpose` | a finalidade: 1 compra, 2 reforma, 31 refinanciamento sem saque, 32 refinanciamento com saque | 1.5, 1.6, 6 |
| `loan_amount`, `property_value` | valor financiado e valor do imóvel, em dólares | 1.2, 1.6 |
| `income` | renda anual, em milhares de dólares (62 quer dizer US$ 62 mil) | 1.2, 1.6 |
| `occupancy_type` | ocupação: moradia do cliente, segunda casa ou investimento | 1.2, 1.6 |
| `lien_status` | posição da hipoteca: primeira ou subordinada | 1.6 |
| `total_units` | número de unidades do imóvel | 1.6 |
| `preapproval` | se o pedido veio de uma pré aprovação | 1.6 |
| `state_code` | estado do imóvel | 1.6 |
| `applicant_credit_score_type` | qual modelo de score o banco usou; o valor do score não é publicado | 1.6, 7.1 |
| `aus-1` | qual sistema automático de aprovação foi usado; o resultado dele não é publicado | 1.6, fora da busca (explicado lá) |
| `lei` | código que identifica o banco (LEI, Legal Entity Identifier) | só identificação; nunca entra na análise nem no agente |

A seção 4 mostra quanto cada uma dessas colunas vem vazia.

**O DTI não vem sempre como número exato.** Para proteger a privacidade, o CFPB publica o DTI em faixas abaixo de 36% (`<20%`, `20%-<30%`, `30%-<36%`) e acima de 49% (`50%-60%`, `>60%`). Só de 36% a 49% o valor vem individual. Por isso as comparações de DTI no notebook usam faixas.

**Pergunta:** o que quer dizer cada código da decisão do banco e do motivo da negativa?

In [3]:
# Códigos do dicionário oficial do CFPB e a tradução usada no projeto
display(pd.Series(ACTION_TAKEN, name='decisão do banco').rename_axis('código'))
pd.Series(DENIAL_REASON, name='motivo da negativa').rename_axis('código')

código
1                             originado
2                  aprovado, não aceito
3                                negado
4                 retirado pelo cliente
5              fechado por incompletude
6                  comprado de terceiro
7                  pré aprovação negada
8    pré aprovação aprovada, não aceita
Name: decisão do banco, dtype: object

código
1                                         DTI
2                        histórico de emprego
3                        histórico de crédito
4                           garantia (imóvel)
5                      recursos insuficientes
6                  informação não verificável
7                         cadastro incompleto
8                   seguro hipotecário negado
9                                       outro
10                              não se aplica
1111    isento (banco dispensado de informar)
Name: motivo da negativa, dtype: object

**Onde olhar:** a coluna de texto das duas tabelas.

**O que se vê:** oito desfechos possíveis para o pedido e dez motivos de negativa, mais o código 1111, usado quando o banco é dispensado de informar o motivo.

**O que significa:** é com esses rótulos que as tabelas seguintes aparecem. O código numérico continua no dado, e a tradução serve só para a leitura.

### 1.1 Como os pedidos terminaram

**Pergunta:** quantos pedidos caem em cada desfecho da coluna `action_taken`?

A coluna não descreve o pedido, e sim como ele terminou. "Originado" quer dizer que o financiamento saiu, ou seja, o banco aprovou, o cliente assinou e o dinheiro foi liberado. Não indica onde o pedido foi feito.

In [4]:
decisoes = (
    hmda['action_taken'].map(ACTION_TAKEN).value_counts()
    .to_frame('pedidos')
)
decisoes['%'] = (decisoes['pedidos'] / len(hmda) * 100).round(1)
decisoes

,pedidos,%
action_taken,,
originado,906942,44.0
comprado de terceiro,427636,20.7
retirado pelo cliente,282798,13.7
negado,265894,12.9
fechado por incompletude,102018,4.9
"aprovado, não aceito",44461,2.2
"pré aprovação aprovada, não aceita",20765,1.0
pré aprovação negada,12420,0.6


**Onde olhar:** as linhas "originado" e "negado".

**O que se vê:** 906.942 originados (44%) e 265.894 negados (12,9%).

**O que significa:** esses dois grupos fazem o papel de gabarito, "o banco aprovou" e "o banco negou". Os outros grupos ficam de lado nesta seção, cada um por um motivo:

- *comprado de terceiro*: quem decidiu foi outro banco;
- *retirado pelo cliente*: o cliente desistiu antes da decisão;
- *fechado por incompletude*: o pedido parou por falta de documento (a seção 4.1 mostra por que sai da comparação);
- *aprovado, não aceito*: o banco aprovou, mas o cliente não fechou, por exemplo porque achou juros menores em outro banco. A decisão do banco foi aprovar, então na preparação dos dados (notebook 02) esse grupo conta como aprovação;
- *pré aprovação negada* e *pré aprovação aprovada, não aceita*: é uma consulta feita antes de o cliente escolher o imóvel, não um pedido de financiamento de um imóvel definido.

A seção 4.1 volta aos fechados por incompletude e aos retirados pelo cliente. Fora o aprovado, não aceito, que conta como aprovação, esses grupos saem da comparação com o banco no notebook 02.

### 1.2 Dez pedidos negados, como chegam na base

**Pergunta:** antes de contar qualquer coisa, como é um pedido negado de verdade?

Dez pedidos negados sorteados com semente fixa. As colunas de motivo vêm primeiro, para aparecerem sem rolar a tabela.

**Por que há três colunas de motivo.** O banco pode informar até quatro motivos para uma mesma negativa, em ordem de importância. O `denial_reason-1` é o principal, o `denial_reason-2` é o segundo, e assim por diante. A primeira tabela conta quantos negados trazem cada um deles. A segunda é a amostra, que mostra só os três primeiros, porque o quarto quase nunca vem preenchido.

In [5]:
negados = hmda[hmda['action_taken'] == '3'].copy()

# Quantos negados trazem o 1º, 2º, 3º e 4º motivo
todos_motivos = [
    'denial_reason-1', 'denial_reason-2', 'denial_reason-3', 'denial_reason-4',
]
preenchidos = negados[todos_motivos].notna().sum().to_frame('negados')
preenchidos['%'] = (preenchidos['negados'] / len(negados) * 100).round(1)
display(preenchidos)

motivo_cols = ['denial_reason-1', 'denial_reason-2', 'denial_reason-3']
cols = motivo_cols + [
    'debt_to_income_ratio', 'loan_to_value_ratio', 'loan_purpose',
    'occupancy_type', 'loan_amount', 'property_value', 'income',
]
amostra = negados.sample(10, random_state=SEED)[cols]
for c in motivo_cols:
    amostra[c] = amostra[c].map(DENIAL_REASON)
amostra

,negados,%
denial_reason-1,265894,100.0
denial_reason-2,53184,20.0
denial_reason-3,7540,2.8
denial_reason-4,559,0.2


,denial_reason-1,denial_reason-2,denial_reason-3,debt_to_income_ratio,loan_to_value_ratio,loan_purpose,occupancy_type,loan_amount,property_value,income
1904803,DTI,NaN,NaN,46,80,32,1,285000.0,355000,62
1282732,garantia (imóvel),outro,NaN,50%-60%,96.5,1,1,255000.0,265000,69
1160625,recursos insuficientes,NaN,NaN,46,96.5,1,1,315000.0,315000,93
267320,histórico de crédito,garantia (imóvel),DTI,50%-60%,86.836,32,1,385000.0,435000,109
251160,DTI,NaN,NaN,NaN,NaN,1,1,315000.0,325000,92
811209,DTI,NaN,NaN,50%-60%,NaN,1,1,325000.0,NaN,56
195470,garantia (imóvel),NaN,NaN,36,56.4,31,1,65000.0,125000,31
947837,histórico de crédito,NaN,NaN,46,80.188,32,1,445000.0,555000,88
209750,garantia (imóvel),NaN,NaN,20%-<30%,68.42,1,1,135000.0,195000,53
210201,garantia (imóvel),NaN,NaN,50%-60%,80,32,1,305000.0,375000,93


**Onde olhar:** na primeira tabela, a coluna %. Na amostra, as três colunas de motivo, depois o DTI (`debt_to_income_ratio`) e o CLTV (`loan_to_value_ratio`). Os pedidos são citados pelo número da linha, à esquerda.

**O que se vê:**

- Todo negado traz o motivo principal. Um em cada cinco traz também um segundo motivo (20,0%), o terceiro aparece em 2,8% e o quarto em 0,2%.
- Na amostra, só dois pedidos têm mais de um motivo, o 1282732 (garantia e outro) e o 267320 (histórico de crédito, garantia e DTI).
- Dos três negados por DTI (1904803, 251160 e 811209), dois têm DTI de 46% ou na faixa de 50% a 60%, e o terceiro vem sem DTI (`NaN`).
- Os quatro negados por garantia (1282732, 195470, 209750 e 210201) têm CLTV entre 56% e 96,5%.
- Os dois negados por histórico de crédito (267320 e 947837) não têm nenhum campo que explique a negativa, porque o HMDA não publica score nem histórico.

**O que significa:** nos negados por DTI, o número publicado explica a negativa, porque o DTI está acima de 43%, o limite da norma para aprovar sem condição extra. Nos negados por garantia, não. O CLTV de todos está no teto de 96,5% ou abaixo dele, então, olhando só os números, nada indicaria problema com o imóvel. Dez pedidos são pouco para concluir; as seções 1.4 e 1.5 repetem a comparação na base inteira. Daqui em diante, a análise usa só o motivo principal (`denial_reason-1`), para cada negativa ser contada uma vez. Como um em cada cinco negados tem segundo motivo, a seção 1.3 conta também quantos negados por um motivo que o agente não enxerga trazem DTI ou garantia em segundo plano.

### 1.3 Por qual motivo os bancos negam

**Pergunta:** de todas as negativas, quantas têm um motivo que o agente consegue enxergar nos números do pedido?

**Por que agrupar os motivos.** O agente só vê os campos publicados. Para cada motivo, a pergunta é se existe na base um campo que mostre o problema. Isso divide os nove motivos do dicionário em três grupos:

- *visível*: DTI e garantia. O banco declarou o motivo, e a base traz o campo ligado a ele (o DTI e o CLTV).
- *invisível*: histórico de crédito, histórico de emprego e informação não verificável. O banco declarou o motivo, mas o dado por trás dele (score, histórico, documentos) não é publicado. O agente não tem como chegar ao motivo declarado. Em parte desses pedidos, chega à mesma decisão por outro caminho, como um DTI muito alto (seção 1.6) ou um segundo motivo visível (contagem abaixo).
- *outros*: cadastro incompleto, recursos insuficientes, seguro hipotecário negado e o próprio código "outro" do dicionário. Nenhum tem um campo que mostre o problema de forma direta, mas também não foram excluídos de antemão.

Há ainda o código 1111, usado quando o banco é dispensado de informar o motivo. Ele não diz motivo nenhum e fica numa linha própria.

Nada é removido aqui, só contado. Quais pedidos invisíveis saem da avaliação é decidido na preparação dos dados (notebook 02). A separação está na função `bloco` da célula abaixo, com as listas `DENIAL_VISIBLE` e `DENIAL_INVISIBLE` de `src/hmda.py`.

A célula mostra duas tabelas, a primeira motivo a motivo e a segunda com os mesmos motivos somados nos grupos. No fim, conta quantos negados do grupo invisível trazem DTI ou garantia como segundo, terceiro ou quarto motivo.

In [6]:
def bloco(code: str) -> str:
    if code in DENIAL_VISIBLE:
        return 'visível (DTI, garantia)'
    if code in DENIAL_INVISIBLE:
        return 'invisível'
    if code == '1111':
        return 'isento (sem motivo informado)'
    return 'outros'

motivos = (
    negados['denial_reason-1'].map(DENIAL_REASON).value_counts()
    .to_frame('negativas')
)
motivos['%'] = (motivos['negativas'] / len(negados) * 100).round(1)

blocos = negados['denial_reason-1'].map(bloco).value_counts().to_frame('negativas')
blocos['%'] = (blocos['negativas'] / len(negados) * 100).round(1)
display(motivos)
display(blocos)

# Negados por motivo invisível que trazem DTI ou garantia como 2º, 3º ou 4º motivo
invisiveis = negados[negados['denial_reason-1'].isin(DENIAL_INVISIBLE)]
secundarios = invisiveis[['denial_reason-2', 'denial_reason-3', 'denial_reason-4']]
tem_visivel = secundarios.isin(DENIAL_VISIBLE).any(axis=1)
print('negados por motivo invisível:', len(invisiveis))
print('com DTI ou garantia como motivo secundário:', tem_visivel.sum())

,negativas,%
denial_reason-1,,
DTI,73816,27.8
histórico de crédito,53720,20.2
garantia (imóvel),44662,16.8
cadastro incompleto,39441,14.8
outro,19785,7.4
recursos insuficientes,16858,6.3
informação não verificável,12477,4.7
histórico de emprego,4525,1.7
isento (banco dispensado de informar),526,0.2


,negativas,%
denial_reason-1,,
"visível (DTI, garantia)",118478,44.6
outros,76168,28.6
invisível,70722,26.6
isento (sem motivo informado),526,0.2


negados por motivo invisível: 70722
com DTI ou garantia como motivo secundário: 8158


**Onde olhar:** na segunda tabela, a linha "visível". Depois, a contagem no fim da célula.

**O que se vê:** 44,6% das negativas têm DTI ou garantia como motivo principal. Na primeira tabela, DTI sozinho é o motivo mais comum (27,8%) e garantia é o terceiro (16,8%). O código 1111 aparece em 526 negados (0,2%). Dos 70.722 negados por motivo invisível, 8.158 trazem DTI ou garantia como motivo secundário.

**O que significa:** a regra dos 25% passa. Mas "visível" aqui quer dizer só que existe um campo com aquele assunto. Falta ver se o campo de fato mostra o problema. Um pedido negado por DTI com DTI de 30% não ajuda o agente.

Os 8.158 têm um motivo que o agente consegue enxergar, só que não em primeiro lugar. Duas decisões ficam para depois. O notebook 02 decide se eles e os invisíveis com DTI acima de 60% saem da avaliação, e a Evaluation decide se o motivo citado pelo agente é comparado só com o principal do banco ou com qualquer um dos quatro.

### 1.4 Quem foi negado por DTI tem mesmo DTI alto? (H1)

**Pergunta:** se o banco nega "por DTI", espera-se que o DTI desses pedidos seja alto. Alto comparado a quê? A quem foi aprovado. Por isso a seção compara dois grupos, lado a lado:

- **negados por DTI:** os 73.816 pedidos cujo motivo principal foi DTI;
- **originados:** os 906.942 pedidos que o banco aprovou e o cliente assinou.

**H1: o DTI publicado explica a negativa por DTI?** São dois montes de pedidos, os negados por DTI e os aprovados. Se o banco olha o DTI para negar, o monte dos negados deve ter muito mais pedidos acima de 43% do que o monte dos aprovados. Se o DTI não importasse, os dois montes teriam mais ou menos a mesma parcela acima de 43%, e olhando só o DTI não daria para dizer quem foi negado. A regra do dobro pede que a parcela dos negados seja pelo menos o dobro da dos aprovados.

As tabelas vão do resumo ao detalhe.

In [7]:
originados = hmda[hmda['action_taken'] == '1']
neg_dti = negados[negados['denial_reason-1'] == '1']

def resumo_dti(df: pd.DataFrame) -> pd.Series:
    acima = dti_above(df['debt_to_income_ratio'])
    return pd.Series({
        'pedidos': len(df),
        'DTI não informado (%)': round(acima.isna().mean() * 100, 1),
        'DTI > 43% entre informados (%)': round(acima.mean() * 100, 1),
    })

pd.DataFrame({
    'negados por DTI': resumo_dti(neg_dti),
    'originados': resumo_dti(originados),
})

,negados por DTI,originados
pedidos,73816.0,906942.0
DTI não informado (%),2.3,10.9
DTI > 43% entre informados (%),92.1,57.2


**Onde olhar:** a última linha, "DTI > 43% entre informados".

**O que se vê:** 92,1% dos negados por DTI passam de 43%. Mas 57,2% dos originados também passam.

**O que significa:** os negados por DTI têm mesmo DTI alto, mas ter DTI acima de 43% não basta para o banco negar, já que mais da metade dos aprovados também tem. A parcela dos negados (92,1%) não chega ao dobro da dos aprovados (57,2%), então a regra do dobro não passa. A seção 1.7 calcula a razão exata. Pedidos sem DTI ficam fora dessa conta, e a linha do meio mostra quantos são.

**Pergunta:** em que faixa de DTI os dois grupos se misturam?

A tabela abre o DTI em todas as faixas publicadas. Cada coluna soma 100% entre os pedidos com DTI informado. Os sem DTI estão na tabela anterior.

In [8]:
# Distribuição completa das faixas de DTI publicadas, nos dois grupos (%)
# O pandas lê o 'NA' do arquivo como vazio, por isso a faixa não entra aqui
faixas_publicadas = [faixa for faixa in DTI_ORDER if faixa != 'NA']
dist = pd.DataFrame({
    'negados por DTI': neg_dti['debt_to_income_ratio']
    .value_counts(normalize=True),
    'originados': originados['debt_to_income_ratio']
    .value_counts(normalize=True),
}).reindex(faixas_publicadas).fillna(0).mul(100).round(1)
dist

,negados por DTI,originados
debt_to_income_ratio,,
<20%,2.8,1.0
20%-<30%,0.7,6.1
30%-<36%,1.0,10.3
36,0.2,2.4
37,0.3,2.6
38,0.3,2.8
39,0.3,3.0
40,0.4,3.2
41,0.5,3.4


**Onde olhar:** as linhas `50%-60%` e `>60%`, perto do fim.

**O que se vê:** na faixa `>60%` estão 52,3% dos negados por DTI e só 0,2% dos originados. Na faixa `50%-60%`, os dois grupos têm quase o mesmo peso (31,1% e 30,0%). Abaixo de 50%, os originados aparecem em todas as faixas.

**O que significa:** só acima de 60% o DTI separa os dois grupos com clareza. E 30% dos aprovados estão na faixa de 50% a 60%, acima do teto de 50% da análise manual.

**Pergunta:** a primeira linha da tabela, `<20%`, tem 2,8% dos negados por DTI. Por que alguém seria negado por dívida alta com DTI abaixo de 20%? Antes de qualquer palpite, uma amostra desses pedidos como chegam na base.

In [9]:
# Amostra bruta dos negados por DTI com DTI abaixo de 20%
# income vem em milhares de dólares
dti_baixo = neg_dti[neg_dti['debt_to_income_ratio'] == '<20%']
colunas_amostra = [
    'debt_to_income_ratio', 'income', 'loan_amount', 'loan_purpose',
    'loan_to_value_ratio', 'denial_reason-2',
]
print('negados por DTI com DTI abaixo de 20%:', len(dti_baixo))
dti_baixo[colunas_amostra].sample(10, random_state=SEED)

negados por DTI com DTI abaixo de 20%: 1992


,debt_to_income_ratio,income,loan_amount,loan_purpose,loan_to_value_ratio,denial_reason-2
207769,<20%,0,45000.0,1,96.5,6
1236246,<20%,-33,455000.0,1,85.0,NaN
54731,<20%,0,315000.0,32,80,NaN
453106,<20%,540,455000.0,31,76.38,3
463821,<20%,216,175000.0,32,43.75,3
215039,<20%,0,375000.0,1,96.5,NaN
1807070,<20%,73,225000.0,1,96.5,2
1969117,<20%,0,185000.0,1,98.190,2
198816,<20%,-22,325000.0,32,70,NaN
947174,<20%,1,245000.0,32,101.75,NaN


**Onde olhar:** a coluna `income`, a renda anual em milhares de dólares.

**O que se vê:** em 6 dos 10 pedidos a renda é zero ou negativa (0, -33, -22), e em mais um é 1, ou seja, US$ 1 mil por ano.

**O que significa:** a renda do cadastro não combina com um DTI publicado abaixo de 20%. Daí vem a hipótese de que renda zero ou negativa torna o DTI publicado sem sentido. A célula seguinte conta quantos dos 1.992 estão nesse caso.

In [10]:
# Renda zero ou negativa (income vem em milhares de dólares)
renda_neg = pd.to_numeric(neg_dti['income'], errors='coerce')
renda_ori = pd.to_numeric(originados['income'], errors='coerce')
neg_20 = neg_dti['debt_to_income_ratio'] == '<20%'
ori_20 = originados['debt_to_income_ratio'] == '<20%'

pd.DataFrame({
    'negados por DTI': [
        neg_20.sum(), (neg_20 & (renda_neg <= 0)).sum(), (renda_neg <= 0).sum(),
    ],
    'originados': [
        ori_20.sum(), (ori_20 & (renda_ori <= 0)).sum(), (renda_ori <= 0).sum(),
    ],
}, index=[
    'DTI abaixo de 20%',
    'DTI abaixo de 20% e renda zero ou negativa',
    'renda zero ou negativa, qualquer DTI',
])

,negados por DTI,originados
DTI abaixo de 20%,1992,8405
DTI abaixo de 20% e renda zero ou negativa,1433,787
"renda zero ou negativa, qualquer DTI",2028,7124


**Onde olhar:** as duas primeiras linhas, na coluna dos negados por DTI.

**O que se vê:** dos 1.992 negados por DTI com DTI abaixo de 20%, 1.433 têm renda zero ou negativa no cadastro. Entre os originados, 7.124 têm renda zero ou negativa, então o problema aparece nos dois grupos.

**O que significa:** com renda zero, a conta do DTI não fecha, e o número publicado não representa a dívida do cliente. A hipótese explica 1.433 dos 1.992. Os outros 559 têm renda positiva ou vazia e pedem outro olhar.

In [11]:
# Os negados por DTI abaixo de 20% que a renda zero não explica
renda_baixo = pd.to_numeric(dti_baixo['income'], errors='coerce')
sem_explicacao = dti_baixo[~(renda_baixo <= 0)]
print('sem renda zero ou negativa:', len(sem_explicacao))
renda_resto = pd.to_numeric(sem_explicacao['income'], errors='coerce')
print(renda_resto.describe().round(0))
sem_explicacao[colunas_amostra].sample(10, random_state=SEED)

sem renda zero ou negativa: 559
count     545.0
mean      128.0
std       313.0
min         1.0
25%        13.0
50%        37.0
75%       111.0
max      3600.0
Name: income, dtype: float64


,debt_to_income_ratio,income,loan_amount,loan_purpose,loan_to_value_ratio,denial_reason-2
503142,<20%,81,75000.0,32,23.077,3
947091,<20%,32,505000.0,32,45.926,NaN
464566,<20%,85,65000.0,31,46.95,3
926389,<20%,1,495000.0,1,98.189,NaN
473963,<20%,159,145000.0,32,66.667,3
939664,<20%,1,455000.0,1,91.573,3
939674,<20%,8,355000.0,1,94.42,NaN
475315,<20%,168,95000.0,32,49,3
1227838,<20%,591,585000.0,31,80.0,6
495220,<20%,3000,255000.0,32,45.273,3


**Onde olhar:** a renda no resumo e, na amostra, as colunas `income` e `denial_reason-2` (o 2º motivo; 3 é histórico de crédito).

**O que se vê:** a contagem de renda (545) fica abaixo dos 559 porque parte vem sem renda. A renda vai de 1 a 3.600, com mediana de 37. Na amostra há dois tipos. Um tem renda quase zero (1 e 8, ou seja, US$ 1 mil e US$ 8 mil por ano). O outro tem renda comum, de US$ 32 mil a US$ 3 milhões, e nele o 2º motivo é histórico de crédito em 5 de 7.

**O que significa:** a renda quase zero é o mesmo problema de cadastro da renda zero, e um filtro só em renda zero ou negativa deixa esses pedidos passarem. No resto, o DTI publicado não sustenta a negativa por DTI, e o histórico de crédito aparece como 2º motivo. O agente não tem como chegar a esses casos pelo DTI. São 559 de 73.816 negados por DTI, e o notebook 02 decide o tratamento da renda zero, negativa ou quase zero e desses casos.

**Pergunta:** e em número de pedidos, quantos aprovados há para cada negado?

Percentual esconde tamanho, porque há muito mais originados (906.942) do que negados por DTI (73.816). A resposta também depende de quem conta como negado. As tabelas acima usam o DTI como motivo principal (1º motivo). Aqui entram mais duas versões, os negados com DTI em qualquer um dos quatro motivos e todos os negados, por qualquer motivo. A tabela junta as faixas em quatro blocos, que seguem a norma.

- até 43%: a norma aprova sem fator compensatório, desde que o PTI fique em até 31%;
- 44% a 49%: a norma exige score e fator compensatório;
- 50% a 60%: no teto da análise manual, que é 50%, ou acima dele;
- acima de 60%: fora de qualquer linha da norma.

A primeira tabela traz as contagens. A segunda divide os originados pelos negados de cada bloco, em cada versão.

In [12]:
BANDS = ['até 43', '44 a 49', '50 a 60', 'acima de 60']
# DTI como 1º, 2º, 3º ou 4º motivo (todos_motivos vem da seção 1.2)
dti_qualquer = negados[(negados[todos_motivos] == '1').any(axis=1)]
grupos_negados = {
    'negados por DTI (1º motivo)': neg_dti,
    'negados por DTI (qualquer motivo)': dti_qualquer,
    'todos os negados': negados,
}

faixas = pd.DataFrame(index=BANDS)
faixas['originados'] = dti_band(originados['debt_to_income_ratio']).value_counts()
for nome, grupo in grupos_negados.items():
    faixas[nome] = dti_band(grupo['debt_to_income_ratio']).value_counts()
display(faixas)

# Originados por negado, com cada definição de negado
razao = pd.DataFrame(index=BANDS)
for nome in grupos_negados:
    razao[nome] = (faixas['originados'] / faixas[nome]).round(2)
razao

,originados,negados por DTI (1º motivo),negados por DTI (qualquer motivo),todos os negados
até 43,345337,5712,6727,78638
44 a 49,217493,6166,7635,39437
50 a 60,243086,22469,26098,59476
acima de 60,1889,37762,41818,51171


,negados por DTI (1º motivo),negados por DTI (qualquer motivo),todos os negados
até 43,60.46,51.34,4.39
44 a 49,35.27,28.49,5.51
50 a 60,10.82,9.31,4.09
acima de 60,0.05,0.05,0.04


**Onde olhar:** a segunda tabela, comparando a primeira coluna com a última.

**O que se vê:** com os negados por DTI como 1º motivo, há 60 aprovados para cada negado até 43%, 35 de 44% a 49% e 11 de 50% a 60%. Com o DTI em qualquer motivo, os números caem pouco (51, 28 e 9). Com todos os negados, a razão fica entre 4 e 6 nas três faixas até 60% (4,39, 5,51 e 4,09). Acima de 60%, os negados são a grande maioria nas três versões (0,04 a 0,05).

**O que significa:** até 60%, o DTI quase nunca é o motivo da negativa, mesmo acima do limite de 43%. Mas isso não quer dizer que o pedido quase sempre é aprovado. Há cerca de 4 a 6 aprovados para cada negado nessas faixas, e a maior parte dessas negativas vem de outros motivos. Só acima de 60% o DTI decide. Quantos negados de cada faixa ficam na avaliação depende de quais motivos saem no notebook 02 (seção 1.3). A seção 1.7 trata da faixa de 44% a 60%.

### 1.5 Quem foi negado por garantia tem mesmo valor financiado alto? (H2)

**Pergunta:** a mesma da 1.4, agora para o outro motivo visível. A garantia é o próprio imóvel, e o banco nega por garantia quando acha que o imóvel não cobre a dívida. O campo ligado a esse motivo é o CLTV (publicado como `loan_to_value_ratio`), que é a dívida dividida pelo valor do imóvel. Num imóvel de US$ 300 mil com US$ 290 mil financiados, o CLTV é 96,7%.

**H2: o CLTV publicado explica a negativa por garantia?** Se o banco olha o valor do imóvel para negar, os negados por garantia deveriam passar do teto de CLTV muito mais vezes que os aprovados. Se o CLTV não importasse, os dois grupos passariam do teto na mesma proporção, e o agente não teria como enxergar esse motivo no dado.

Primeiro, os dois grupos inteiros, sem separar por finalidade.

In [13]:
neg_gar = negados[negados['denial_reason-1'] == '4']

def resumo_cltv(df: pd.DataFrame) -> pd.Series:
    cltv = pd.to_numeric(df['loan_to_value_ratio'], errors='coerce')
    return pd.Series({
        'pedidos': len(df),
        'CLTV não informado (%)': round(cltv.isna().mean() * 100, 1),
        'CLTV mediano': cltv.median(),
    })

pd.DataFrame({
    'negados por garantia': resumo_cltv(neg_gar),
    'originados': resumo_cltv(originados),
})

,negados por garantia,originados
pedidos,44662.0000,906942.0
CLTV não informado (%),4.3000,7.3
CLTV mediano,85.8385,96.5


**Onde olhar:** a linha "CLTV mediano". A mediana é o valor do meio, com metade dos pedidos abaixo dela e metade acima.

**O que se vê:** a mediana é 85,8% nos negados por garantia e 96,5% nos originados. Os negados têm CLTV *menor*.

**O que significa:** olhando os grupos inteiros, o resultado vai na direção contrária à esperada. Antes de concluir algo, vale ver de que tipo de pedido cada grupo é feito.

**Pergunta:** de que finalidade são os pedidos de cada grupo?

In [14]:
pd.DataFrame({
    'negados por garantia (%)': neg_gar['loan_purpose'].map(LOAN_PURPOSE)
    .value_counts(normalize=True),
    'originados (%)': originados['loan_purpose'].map(LOAN_PURPOSE)
    .value_counts(normalize=True),
}).mul(100).round(1).sort_values('originados (%)', ascending=False)

,negados por garantia (%),originados (%)
loan_purpose,,
compra,23.8,70.6
refinanciamento com saque,56.2,15.2
refinanciamento sem saque,14.5,12.6
outro,4.3,1.2
reforma,1.1,0.3
não se aplica,0.1,0.0


**Onde olhar:** as linhas "compra" e "refinanciamento com saque".

**O que se vê:** 56,2% dos negados por garantia são refinanciamento com saque, contra 15,2% dos originados. A compra é 70,6% dos originados e só 23,8% dos negados por garantia.

**O que significa:** a mediana de cada grupo reflete a finalidade que domina o grupo, e não o risco. No refinanciamento com saque, o cliente já pagou parte do imóvel e o teto é 80%. Na compra, a entrada mínima é 3,5%, e o CLTV fica perto de 96,5%. Os originados, na maioria compras, ficam com mediana alta; os negados por garantia, na maioria saques, ficam com mediana mais baixa. Um CLTV de 85% está dentro do teto numa compra e fora dele num saque. Por isso a comparação precisa ser feita finalidade a finalidade, com cada pedido contra o teto da sua.

**Pergunta:** separando por finalidade, quantos pedidos passam do teto da norma?

Os tetos de CLTV da norma, resumidos na abertura:

| Finalidade | Teto de CLTV | Seção do Handbook |
|---|---|---|
| compra | 96,5% | II.A.2 |
| refinanciamento sem saque | 97,75%, exceto o Streamline, que não tem teto | II.A.8 |
| refinanciamento com saque | 80% | II.A.8 |

O HMDA não diz qual dos três tipos de refinanciamento sem saque é cada pedido. A linha dessa finalidade mistura pedidos com teto de 97,75% e pedidos Streamline, sem teto.

In [15]:
# Teto por finalidade: compra, refinanciamento sem saque e com saque
from src.rules import CLTV_CAP

def acima_do_teto(df: pd.DataFrame) -> pd.DataFrame:
    d = df[df['loan_purpose'].isin(CLTV_CAP.keys())]
    cltv = pd.to_numeric(d['loan_to_value_ratio'], errors='coerce')
    acima = cltv > d['loan_purpose'].map(CLTV_CAP)
    return pd.DataFrame({
        'pedidos': d.groupby('loan_purpose').size(),
        'acima do teto (%)': acima.groupby(d['loan_purpose']).mean()
        .mul(100).round(1),
    }).rename(index=LOAN_PURPOSE)

pd.concat({
    'negados por garantia': acima_do_teto(neg_gar),
    'originados': acima_do_teto(originados),
}, axis=1)

negados por garantia                   originados  \
                                       pedidos acima do teto (%)    pedidos   
loan_purpose                                                                  
compra                                   10647              38.7     640557   
refinanciamento sem saque                 6454              31.9     114566   
refinanciamento com saque                25096              40.6     137966   

                                             
                          acima do teto (%)  
loan_purpose                                 
compra                                 21.3  
refinanciamento sem saque               1.3  
refinanciamento com saque               1.7

**Onde olhar:** a coluna "acima do teto (%)" dos dois grupos, linha a linha.

**O que se vê:** no refinanciamento com saque, 40,6% dos negados por garantia passam do teto de 80%, contra 1,7% dos originados. No refinanciamento sem saque, 31,9% contra 1,3%. Na compra, 38,7% contra 21,3%.

**Resposta à H2.** Em parte. Nos dois refinanciamentos, passar do teto é raro entre os aprovados e comum entre os negados por garantia. Ali o motivo aparece no dado, e o agente tem como perceber e citar a seção da norma. É o tipo de caso em que uma regra fixa de DTI não enxerga nada. No refinanciamento sem saque isso vale mesmo com o Streamline misturado na linha. Por outro lado, a maior parte dos negados por garantia fica dentro do teto em todas as finalidades, já que a parte acima dele não passa de 41% em nenhuma. O banco viu um problema que não aparece nos números, provavelmente no laudo de avaliação do imóvel. Na compra, o sinal é fraco, porque 21,3% dos aprovados também passam de 96,5%. A próxima tabela procura o porquê.

**Pergunta:** as compras aprovadas acima de 96,5% estão fora da norma?

A abertura citou dois casos em que a norma deixa a dívida de uma compra passar de 96,5%. Um é o seguro financiado, que leva a dívida a até 98,2% do imóvel (96,5% somado ao seguro de 1,75%). O outro é o segundo financiamento de órgão público, sem teto. Se o seguro explicasse tudo, essas compras ficariam até 98,2%. A célula divide as compras acima de 96,5% em três faixas e lista os valores de CLTV mais frequentes entre as aprovadas.

In [16]:
def compras_acima(df: pd.DataFrame) -> pd.Series:
    compra = df[df['loan_purpose'] == '1']
    cltv = pd.to_numeric(compra['loan_to_value_ratio'], errors='coerce')
    return pd.cut(
        cltv[cltv > 96.5], bins=[96.5, 98.2, 100, float('inf')],
        labels=['96,5 a 98,2', '98,2 a 100', 'acima de 100'],
    ).value_counts()

tab = pd.DataFrame({
    'negados por garantia': compras_acima(neg_gar),
    'originados': compras_acima(originados),
})
tab['originados (%)'] = (tab['originados'] / tab['originados'].sum() * 100).round(1)
display(tab)

cltv_compra = pd.to_numeric(
    originados.loc[originados['loan_purpose'] == '1', 'loan_to_value_ratio'],
    errors='coerce',
)
print('maior CLTV entre as compras aprovadas:', cltv_compra.max())
print('compras aprovadas com CLTV acima de 150:', (cltv_compra > 150).sum())
print('96,5% somado ao seguro de 1,75% (96,5 x 1,0175):', round(96.5 * 1.0175, 2))

# CLTV mais frequentes nas compras aprovadas acima de 96,5%, com duas casas
# decimais (o HMDA publica 98,189 e 98,188 para o mesmo cálculo)
cltv_compra[cltv_compra > 96.5].round(2).value_counts().head(6)\
    .rename('originados')

,negados por garantia,originados,originados (%)
loan_to_value_ratio,,,
acima de 100,2767,69448,50.9
"98,2 a 100",825,48379,35.4
"96,5 a 98,2",529,18658,13.7


maior CLTV entre as compras aprovadas: 560.438
compras aprovadas com CLTV acima de 150: 22
96,5% somado ao seguro de 1,75% (96,5 x 1,0175): 98.19


loan_to_value_ratio
100.00    28561
101.50    14306
100.43    13768
98.19     12533
101.41     4600
99.45      4273
Name: originados, dtype: int64

**Onde olhar:** a coluna "originados (%)", as três linhas impressas e, depois, a lista de valores mais frequentes.

**O que se vê:** só 13,7% das compras aprovadas acima de 96,5% ficam até 98,2%; 50,9% passam de 100%. Na lista, o 98,19 aparece com 12.533 compras, e os outros valores mais comuns são 100, 101,5 e 100,43. No topo, o maior CLTV de uma compra aprovada é 560,4%, e 22 compras passam de 150%.

**O que significa:** 98,19 é exatamente 96,5% somado ao seguro de 1,75% (96,5 x 1,0175). O seguro financiado aparece no dado, mas explica só uma parte. A maior parte passa de 100%, o que a norma só permite com segundo financiamento de órgão público ou de entidade sem fins lucrativos aprovada pelo HUD, comum em programas de ajuda para a entrada (II.A.5, Secondary Financing). O HMDA não publica se o pedido tem segundo financiamento; a seção 6 testa essa explicação na NSMO, que pergunta isso ao cliente. Para o agente, isso quer dizer que uma compra com CLTV acima de 96,5% não basta para negar. Falta saber se há seguro financiado ou segundo financiamento, o que tende a virar pendência, e não negativa. As 22 compras acima de 150% fogem muito dos valores comuns e ficam como candidatas a erro de cadastro, com tratamento decidido no notebook 02.

Falta um último ponto do CLTV, os pedidos aprovados que chegam sem ele.

**Pergunta:** quantos aprovados chegam sem CLTV, em cada finalidade?

O guia oficial de preenchimento do HMDA (*Filing Instructions Guide*, do CFPB) prevê dois códigos no lugar do número. O "NA" é usado quando a exigência de informar o campo não se aplica ao pedido, e o "Exempt" quando o banco tem dispensa parcial e não é obrigado a publicar o campo. A célula conta os dois juntos.

In [17]:
# Ao ler o CSV, o pandas transforma o texto 'NA' em vazio; 'Exempt' continua texto
fin = originados[originados['loan_purpose'].isin(['1', '31', '32'])]
cltv_fin = fin['loan_to_value_ratio']
sem_cltv = cltv_fin.isna() | (cltv_fin == 'Exempt')
vazio_fin = sem_cltv.groupby(fin['loan_purpose'].map(LOAN_PURPOSE))\
    .agg(['size', 'sum', 'mean'])
vazio_fin.columns = ['originados', 'sem CLTV', 'sem CLTV (%)']
vazio_fin['sem CLTV (%)'] = (vazio_fin['sem CLTV (%)'] * 100).round(1)
vazio_fin

,originados,sem CLTV,sem CLTV (%)
loan_purpose,,,
compra,640557,9709,1.5
refinanciamento com saque,137966,1774,1.3
refinanciamento sem saque,114566,47669,41.6


**Onde olhar:** a coluna "sem CLTV (%)".

**O que se vê:** 41,6% dos refinanciamentos sem saque aprovados chegam sem CLTV (47.669 pedidos), contra 1,5% das compras e 1,3% dos saques.

**O que significa:** no refinanciamento sem saque, a ausência é coerente com o Streamline, que dispensa a avaliação do imóvel e por isso não tem CLTV para informar. Para o agente, sem o número, a resposta fiel à norma é suspender, e isso discorda do banco, que aprovou. Por isso a concordância desses pedidos é reportada à parte na avaliação, e o tratamento deles é decidido na preparação dos dados.

**Decisão:** os negados por garantia ficam na avaliação, com o resultado reportado à parte por finalidade. Tirar só os que estão dentro do teto seria deixar na prova apenas os casos em que o dado concorda com a resposta, o que inflaria a nota do agente.

### 1.6 Existe outra combinação de campos que denuncie a negativa?

**Pergunta:** a combinação de finalidade com CLTV só apareceu quando os dois campos foram olhados juntos. Existe outra assim, em qualquer motivo, que ainda não foi vista?

Se aparecer outra, o agente ganha mais um caso que consegue enxergar e a Checagem 1 muda. Se não aparecer, o resultado da 1.5 fica como o único sinal combinado do HMDA.

Procurar à mão, par por par, não escala. A busca usa duas medidas:

- **lift:** quantas vezes a combinação é mais comum entre os negados do que entre os aprovados. Se um par aparece em 10% dos negados e em 0,2% dos aprovados, o lift é 50.
- **ganho:** o lift do par dividido pelo maior lift dos dois campos sozinhos. Ganho acima de 1 quer dizer que a combinação conta algo que nenhum dos dois campos conta sozinho. É isso que a busca procura.

A célula calcula as duas medidas para todos os pares de campos que descrevem o pedido e para todos os motivos de negativa, com a função `pair_scan` de `src/explore.py`. Os campos numéricos viram faixas antes (função `binned`, do mesmo arquivo). Um CLTV de 85,3 vira "80-90", e o vazio vira "vazio". Entram só combinações com pelo menos 300 negados e 1% do motivo, para não destacar casos raros. Ficam fora os campos de raça, sexo e idade, que nunca podem pesar numa decisão de crédito, e o `aus-1`, pelo motivo que a próxima célula mostra. Os campos aparecem com nome em português e os códigos traduzidos. A tabela mostra, por motivo, as três combinações de maior ganho entre as que têm lift de pelo menos 3.

**Pergunta:** o código 6 do `aus-1` ("não se aplica", nenhum sistema automático usado) aparece em que tipo de pedido?

In [18]:
# Pedidos sem sistema automático de aprovação (aus-1 = 6), por decisão do banco
decisao = hmda['action_taken'].map(ACTION_TAKEN)
sem_aus = hmda['aus-1'] == '6'
coluna = 'sem sistema automático (aus-1 = 6)'
tabela_aus = decisao.value_counts().to_frame('pedidos')
tabela_aus[coluna] = decisao[sem_aus].value_counts()
tabela_aus[coluna] = tabela_aus[coluna].fillna(0).astype(int)
tabela_aus['%'] = (tabela_aus[coluna] / tabela_aus['pedidos'] * 100).round(1)
tabela_aus

,pedidos,sem sistema automático (aus-1 = 6),%
action_taken,,,
originado,906942,96206,10.6
comprado de terceiro,427636,427632,100.0
retirado pelo cliente,282798,93520,33.1
negado,265894,72126,27.1
fechado por incompletude,102018,49993,49.0
"aprovado, não aceito",44461,16365,36.8
"pré aprovação aprovada, não aceita",20765,646,3.1
pré aprovação negada,12420,2502,20.1


**Onde olhar:** as linhas "originado", "negado" e "comprado de terceiro".

**O que se vê:** o código 6 aparece em 96.206 originados (10,6%), em 72.126 negados (27,1%) e em praticamente todos os comprados de terceiro (427.632 de 427.636), que não passam pelo sistema do banco que comprou.

**O que significa:** o código não marca pedido que parou antes da análise, porque um em cada dez aprovados também o tem. Pode ser pedido decidido por um analista sem passar pelo sistema, refinanciamento simplificado, que dispensa análise, ou outra situação. O HMDA não diz qual. Como o significado não está claro e o campo descreve o processo do banco, e não o pedido do cliente, ele fica fora da busca. O notebook 02 investiga o que o código marca, porque, se forem decisões manuais, esses pedidos são os mais próximos do caso que o projeto quer apoiar.

A tabela também mostra um limite do HMDA. Ele não diz se o pedido foi decidido pelo sistema automático ou por um analista, então a avaliação no HMDA mistura os dois. A NSMO tem o mesmo limite (seção 5.2).

In [19]:
from src.explore import binned, pair_scan
from src.hmda import CREDIT_SCORE_TYPE, LIEN, OCCUPANCY, PREAPPROVAL

CAMPOS_PEDIDO = [
    'loan_purpose', 'lien_status', 'occupancy_type', 'total_units',
    'loan_amount', 'property_value', 'income', 'debt_to_income_ratio',
    'loan_to_value_ratio', 'applicant_credit_score_type', 'preapproval',
    'state_code',
]
# Negados com motivo do dicionário (fora o código 1111, banco dispensado)
decid = hmda[(hmda['action_taken'] == '1') | (
    (hmda['action_taken'] == '3')
    & hmda['denial_reason-1'].isin(DENIAL_REASON.keys())
    & (hmda['denial_reason-1'] != '1111')
)]
grupo = decid['denial_reason-1'].where(decid['action_taken'] == '3', 'originado')

# Nomes em português e códigos traduzidos, só para a leitura da tabela
NOMES = {
    'loan_purpose': 'finalidade', 'lien_status': 'hipoteca',
    'occupancy_type': 'ocupação', 'total_units': 'unidades',
    'loan_amount': 'valor financiado', 'property_value': 'valor do imóvel',
    'income': 'renda', 'debt_to_income_ratio': 'DTI',
    'loan_to_value_ratio': 'CLTV', 'applicant_credit_score_type': 'modelo de score',
    'preapproval': 'pré aprovação', 'state_code': 'estado',
}
campos = binned(decid, CAMPOS_PEDIDO)
for col, labels in [
    ('loan_purpose', LOAN_PURPOSE), ('occupancy_type', OCCUPANCY),
    ('lien_status', LIEN), ('preapproval', PREAPPROVAL),
    ('applicant_credit_score_type', CREDIT_SCORE_TYPE),
]:
    campos[col] = campos[col].map(labels).fillna(campos[col])
campos = campos.rename(columns=NOMES)
pares = pair_scan(campos, list(NOMES.values()), grupo, DENIAL_REASON)
print('pares de campos testados:', len(CAMPOS_PEDIDO) * (len(CAMPOS_PEDIDO) - 1) // 2)
(
    pares[(pares['lift'] >= 3) & (pares['ganho'] > 1)]
    .sort_values(['motivo', 'ganho'], ascending=[True, False])
    .groupby('motivo').head(3)
    .set_index(['motivo', 'combinação'])
)

pares de campos testados: 66


negados  \
motivo                     combinação                                                    
DTI                        valor financiado = 400-500k E renda = 40-60k            860   
                           valor financiado = >500k E renda = 80-100k             1122   
                           valor financiado = 300-400k E renda = 40-60k           3824   
cadastro incompleto        valor do imóvel = vazio E DTI = 50%-60%                 701   
                           finalidade = refinanciamento com saque E valor ...     2816   
                           finalidade = refinanciamento com saque E DTI = ...     6974   
garantia (imóvel)          finalidade = refinanciamento com saque E CLTV =...     2297   
                           finalidade = refinanciamento com saque E CLTV =...     2401   
                           finalidade = refinanciamento sem saque E CLTV =...     1463   
histórico de crédito       valor do imóvel = vazio E modelo de score = Equ...      684   
                           finalidade = refinanciamento com saque E modelo...     1152   
                           finalidade = refinanciamento com saque E CLTV =...     1277   
histórico de emprego       DTI = >60% E CLTV = 90-96,5                             407   
                           finalidade = compra E DTI = >60%                        601   
                           finalidade = compra E renda = 0-40k                     432   
informação não verificável finalidade = refinanciamento com saque E CLTV =...      326   
                           DTI = >60% E CLTV = 90-96,5                             468   
                           finalidade = outro E modelo de score = Experian...      311   
outro                      finalidade = refinanciamento sem saque E estado...      443   
                           DTI = >60% E CLTV = 90-96,5                             435   
                           finalidade = refinanciamento sem saque E DTI = ...      355   
recursos insuficientes     renda = vazio E modelo de score = Equifax Beaco...     1114   
                           CLTV = vazio E modelo de score = Experian Fair ...     1031   
                           DTI = vazio E modelo de score = Equifax Beacon 5.0     1130   

                                                                               originados  \
motivo                     combinação                                                       
DTI                        valor financiado = 400-500k E renda = 40-60k               544   
                           valor financiado = >500k E renda = 80-100k                1460   
                           valor financiado = 300-400k E renda = 40-60k              2879   
cadastro incompleto        valor do imóvel = vazio E DTI = 50%-60%                     20   
                           finalidade = refinanciamento com saque E valor ...         248   
                           finalidade = refinanciamento com saque E DTI = ...        2578   
garantia (imóvel)          finalidade = refinanciamento com saque E CLTV =...          14   
                           finalidade = refinanciamento com saque E CLTV =...         203   
                           finalidade = refinanciamento sem saque E CLTV =...         468   
histórico de crédito       valor do imóvel = vazio E modelo de score = Equ...         887   
                           finalidade = refinanciamento com saque E modelo...        1477   
                           finalidade = refinanciamento com saque E CLTV =...        1945   
histórico de emprego       DTI = >60% E CLTV = 90-96,5                                236   
                           finalidade = compra E DTI = >60%                           699   
                           finalidade = compra E renda = 0-40k                      19363   
informação não verificável finalidade = refinanciamento com saque E CLTV =...        1945   
                           DTI = >60% E CLTV = 90-96,5          

**Onde olhar:** primeiro o bloco da garantia, depois a coluna "ganho" dos outros motivos.

**O que se vê:** no topo da garantia estão as combinações de refinanciamento com saque e CLTV acima do teto, como saque com CLTV acima de 100% (2.297 negados contra 14 originados). Logo abaixo, o refinanciamento sem saque com CLTV acima de 100% (1.463 negados contra 468 originados). Em histórico de emprego, informação não verificável e outro, as combinações têm DTI acima de 60%. Nos demais motivos aparecem combinações de campos vazios ou de empréstimo alto com renda baixa, que é o próprio DTI visto pelos seus ingredientes.

**O que significa:** a busca reencontrou sozinha a combinação da seção 1.5 e achou só mais uma regra visível, e do mesmo tipo, o refinanciamento sem saque com dívida maior que o imóvel, acima do teto de 97,75%. Os 468 aprovados acima de 100% são coerentes com o Streamline, que não tem teto e não vem separado no HMDA, então o agente precisa conhecer essa modalidade antes de negar. O DTI acima de 60% em negativas por outro motivo afeta a avaliação do motivo. Nesses pedidos, o agente tende a negar citando DTI enquanto o banco escreveu outro motivo, e a decisão bate, mas o motivo não. Nenhuma outra regra escondida apareceu.

### 1.7 Resultado da Checagem 1

A célula aplica as duas regras definidas no início da seção.

In [20]:
visivel = negados['denial_reason-1'].isin(DENIAL_VISIBLE).mean()
p_neg = dti_above(neg_dti['debt_to_income_ratio']).mean()
p_ori = dti_above(originados['debt_to_income_ratio']).mean()
razao = p_neg / p_ori

print(f'1. regra dos 25%: motivo visível em {visivel:.1%} -> '
      f'{"passa" if visivel >= 0.25 else "não passa"}')
print(f'2. regra do dobro: DTI > 43% em {p_neg:.1%} dos negados por DTI e '
      f'{p_ori:.1%} dos originados, razão {razao:.2f} -> '
      f'{"passa" if razao >= 2 else "não passa"}')

1. regra dos 25%: motivo visível em 44.6% -> passa
2. regra do dobro: DTI > 43% em 92.1% dos negados por DTI e 57.2% dos originados, razão 1.61 -> não passa


**Onde olhar:** as duas linhas impressas, cada uma com o veredito no fim.

**O que se vê:** a regra dos 25% passa (44,6%) e a regra do dobro não passa (1,61).

**Resposta à H1.** Sim, o DTI publicado tem relação com a negativa por DTI, mas só nas pontas. Até 43% o DTI quase nunca é o motivo da negativa, e acima de 60% quase todos são negados. Entre 44% e 60%, a maioria foi aprovada mesmo acima do limite de 43%. Até 60%, as negativas que existem vêm sobretudo de outros motivos (seção 1.4, todos os negados). O motivo provável é que o sistema automático do FHA (TOTAL Scorecard, um AUS) aprova esses casos quando score e reserva compensam, e o HMDA não publica nenhum dos dois.

**Ressalva na regra dos 25%.** A garantia é visível só em parte. Aparece com clareza no refinanciamento acima do teto (com ou sem saque), de forma fraca na compra, e fica invisível na maior parte dos casos, que ficam dentro do teto da sua finalidade (seção 1.5). O DTI sozinho já soma 27,8% das negativas, então a regra passa mesmo sem contar a garantia, mas por pouco.

Como o HMDA não serve para medir o agente da mesma forma em todos os pedidos, a forma de medir muda.

**Decisão: ajuste na métrica.** O teste principal passa a ser reportado por faixa de DTI, com papel diferente em cada uma:

| Faixa | O que o HMDA mostra | O que se mede |
|---|---|---|
| até 43% | DTI quase nunca é o motivo; as negativas vêm de outros campos | se o agente aprova quando nada mais pesa, e se nega pelo motivo certo quando a negativa vem de um campo visível (garantia) |
| 44 a 60% | decisão depende de score e reserva, ausentes | no HMDA, se o agente reconhece a falta e suspende pedindo os dados; na NSMO, com score e reserva, se aplica a norma (seção 7.2) |
| acima de 60% | quase todos negados | se o agente nega e cita o motivo certo |

Depois da leitura da norma, a seção 5.2 divide a faixa de 44% a 60% no teto manual de 50%.

Limitação aceita. Na faixa de 44 a 60%, nenhuma base pública testa se o agente evita aprovar quem devia ser negado, porque a NSMO só traz aprovados. As negativas por garantia são reportadas à parte, por finalidade.

## 2. Checagem 2: a NSMO cobre a faixa do meio com score e reserva?

**A pergunta.** A Checagem 1 mostrou que, de 44% a 60% de DTI, a decisão do banco depende de score e reserva, que o HMDA não publica. Essa é a faixa do meio, nome usado daqui em diante. A NSMO entra para testar essa faixa. Para isso, um pedido FHA da NSMO precisa trazer três coisas que o analista usaria na mesa, que são o DTI exato (47% e 55% pedem regras diferentes na norma), o score exato (o corte de 580 separa o que é permitido) e a resposta sobre reserva, que é fator compensatório. Se a base vier em faixas largas, ou se poucos casos da faixa tiverem as três informações, o segundo teste não se sustenta.

**Como ler a NSMO.** Cada linha é um financiamento já concedido; a base não tem negados. Pelo codebook (Appendix C, v70), a reserva (`x5001`) vem em quatro categorias (menos de 1 mês, 1 a 2 meses, 3 a 6 meses, 7 meses ou mais de prestação) e só foi perguntada nas ondas 25 a 46 da pesquisa. Códigos negativos marcam ausência (-2 não se aplica, -3 não perguntado na onda, -4 ausente).

A NSMO é uma pesquisa por amostra. Cada linha tem um peso (`analysis_weight`, que este notebook não carrega) que diz quantos financiamentos do país ela representa. O notebook conta linhas, sem peso, porque o agente é avaliado pedido a pedido. Por isso os percentuais da NSMO que aparecem aqui descrevem a amostra, e não o mercado americano.

**Duas regras para responder,** definidas antes de olhar os dados:

1. **Regra dos valores exatos:** entre os FHA, pelo menos 30 valores distintos de DTI e 100 de score. Um arquivo em faixas teria no máximo uma dezena de cada.
2. **Regra dos 400 casos:** pelo menos 400 pedidos FHA com DTI de 44% a 60%, score e reserva informados. Na preparação dos dados, os casos são sorteados em dois grupos. O grupo de ajuste é usado para testar e ajustar o agente durante o desenvolvimento. O grupo da nota final fica guardado e roda uma vez só, na avaliação, e é dele que sai a nota do agente. Com a metade para cada grupo, sobram cerca de 200 para a nota. A margem de erro que esse tamanho dá é calculada na seção 7.2.

> The opinions and analyses contained herein are solely of the users/authors of any data analyses or papers, and the FHFA cannot and does not attest to nor vouch for the quality, accuracy, or timeliness of the data, or analyses derived from these data after the data has been retrieved from FHFA.gov.

In [21]:
from src.nsmo import LOAN_TYPE_FHA, NSMO_CSV, NSMO_TERMS, X5001, load_nsmo

assert NSMO_TERMS.exists(), 'cópia dos Terms of Service da FHFA ausente'
nsmo = load_nsmo()
fha = nsmo[nsmo['loan_type'] == LOAN_TYPE_FHA].copy()
print(NSMO_CSV.name, nsmo.shape, '| FHA:', len(fha))

nsmo_v70_1324_puf.csv (62359, 16) | FHA: 7331


### 2.1 Dez pedidos FHA, como chegam na base

**Pergunta:** como DTI, score e reserva aparecem de fato no arquivo?

In [22]:
fha.sample(10, random_state=SEED)

,nsmoid,survey_wave,x36c,x36d,x5001,open_year,ltv,cltv,dti,pti,loan_type,score_orig_r,score_orig_s,score_orig_o1,score_orig_o2,score_orig_o3
20423,720424,12,2,2,-3,2016,96,96,41,25,2,648,-2,-2,-2,-2
13177,713178,9,-2,-2,-3,2015,88,88,56,41,2,637,643,-2,-2,-2
7379,707380,5,2,-3,-3,2014,96,96,28,15,2,681,730,-2,-2,-2
34588,734589,25,-2,-2,1,2019,93,93,38,18,2,663,-2,-2,-2,-2
15928,715929,8,2,2,-3,2015,87,87,33,23,2,709,-2,-2,-2,-2
59110,759111,43,2,2,3,2024,86,86,56,39,2,798,-2,-2,-2,-2
44107,744108,27,-2,-2,3,2020,67,67,53,21,2,655,659,-2,-2,-2
31814,731815,19,2,2,-3,2018,96,96,54,39,2,714,-2,-2,-2,-2
34766,734767,25,-2,-2,2,2019,86,86,46,23,2,617,577,-2,-2,-2
53600,753601,36,-2,-2,1,2022,68,68,29,25,2,688,-2,-2,-2,-2


**Onde olhar:** as colunas `dti`, `score_orig_r` (score na contratação) e `x5001` (reserva).

**O que se vê:** DTI e score vêm como números quebrados, como 41, 56, 648 e 709, e não em bordas de faixa. A reserva vem como -3 em cinco dos dez pedidos, porque a pergunta não foi feita na onda daquele pedido.

**O que significa:** a amostra sugere valores exatos e reserva faltando em boa parte da base. As próximas tabelas medem as duas coisas.

### 2.2 DTI e score vêm como valor exato?

**Pergunta:** se o arquivo trouxesse só faixas, haveria poucos valores distintos, todos em bordas de faixa. Quantos valores distintos há?

In [23]:
for col in ['dti', 'score_orig_r']:
    valid = fha.loc[fha[col] > 0, col]
    print(f'{col}: {valid.nunique()} valores distintos, '
          f'faixa {valid.min()} a {valid.max()}, '
          f'ausentes ou códigos negativos: {(fha[col] <= 0).sum()}')
    print('  mais frequentes:', valid.value_counts().head(8).to_dict())

# Pedidos no topo da escala do DTI
topo = fha.loc[fha['dti'] >= 95, 'dti'].value_counts().sort_index()
print('DTI de 95 a 99:', topo.to_dict())

dti: 87 valores distintos, faixa 7 a 99, ausentes ou códigos negativos: 0
  mais frequentes: {45: 291, 42: 286, 43: 281, 49: 279, 44: 273, 41: 273, 39: 269, 50: 266}
score_orig_r: 326 valores distintos, faixa 463 a 838, ausentes ou códigos negativos: 53
  mais frequentes: {734: 54, 718: 54, 651: 54, 647: 53, 653: 53, 648: 51, 667: 51, 664: 51}
DTI de 95 a 99: {95: 2, 96: 1, 97: 1, 98: 1, 99: 36}


**Onde olhar:** o número de valores distintos de cada campo.

**O que se vê:** 87 valores de DTI (de 7 a 99) e 326 de score (de 463 a 838). Os mais frequentes são números comuns, como 45 e 42 no DTI, e não bordas de faixa.

No topo do DTI, 99 aparece em 36 pedidos, contra um ou dois em cada valor de 95 a 98.

**O que significa:** DTI e score vêm exatos. As faixas das tabelas do codebook são só a forma de apresentar os resumos. A regra dos valores exatos passa. A exceção é o 99, que o codebook descreve como teto de codificação ("top coded"), que junta todo DTI de 99% ou mais. Os 36 casos estão acima de 50%, fora da faixa avaliada.

**Pergunta:** os campos da NSMO combinam entre si? Duas combinações seriam impossíveis. O PTI (só a prestação) não pode passar do DTI (a prestação mais as outras dívidas), e o LTV (só o primeiro financiamento) não pode passar do CLTV (todas as dívidas sobre o imóvel). Se aparecerem, o notebook 02 precisa de um filtro para elas. Se não, nada muda.

In [24]:
com_dti_pti = fha[(fha['dti'] > 0) & (fha['pti'] > 0)]
pti_maior = com_dti_pti['pti'] > com_dti_pti['dti']
print('PTI maior que DTI:', pti_maior.sum(), 'de', len(com_dti_pti))

com_ltv_cltv = fha[(fha['ltv'] > 0) & (fha['cltv'] > 0)]
ltv_maior = com_ltv_cltv['ltv'] > com_ltv_cltv['cltv']
print('LTV maior que CLTV:', ltv_maior.sum(), 'de', len(com_ltv_cltv))

PTI maior que DTI: 0 de 7326
LTV maior que CLTV: 0 de 7331


**Onde olhar:** as duas contagens.

**O que se vê:** nenhum pedido tem PTI maior que DTI (0 de 7.326), e nenhum tem LTV maior que CLTV (0 de 7.331).

**O que significa:** os campos combinam entre si, e o notebook 02 não precisa de filtro para essas combinações.

### 2.3 Quantos FHA responderam sobre reserva, e em que faixa de DTI?

**Pergunta:** na faixa de 44% a 60%, quantos pedidos têm resposta sobre reserva?

In [25]:
fha['faixa_dti'] = pd.cut(
    fha['dti'], bins=[0, 43, 60, 100],
    labels=['até 43', '44 a 60', 'acima de 60'],
)
fha['reserva'] = fha['x5001'].map(X5001).fillna('sem resposta')
display(pd.crosstab(
    fha['faixa_dti'], fha['reserva'], margins=True, margins_name='total'
))

faixa_meio = fha[fha['faixa_dti'] == '44 a 60']
print('FHA de 44 a 60% com resposta sobre reserva:',
      (faixa_meio['reserva'] != 'sem resposta').sum())

reserva,1 a 2 meses,3 a 6 meses,7 meses ou mais,menos de 1 mês,sem resposta,total
faixa_dti,,,,,,
até 43,479,304,213,252,2770,4018
44 a 60,506,334,191,321,1837,3189
acima de 60,1,5,0,1,117,124
total,986,643,404,574,4724,7331


FHA de 44 a 60% com resposta sobre reserva: 1352


**Onde olhar:** a linha "44 a 60" e a contagem abaixo da tabela.

**O que se vê:** dos 3.189 FHA nessa faixa, 1.352 responderam sobre reserva e 1.837 não.

**O que significa:** a reserva falta em mais da metade da faixa, como previsto pelas ondas, mas os que responderam ainda são muitos.

**Pergunta:** quantos desses também têm score, as três informações juntas que a regra dos 400 casos pede?

In [26]:
meio = fha[
    (fha['faixa_dti'] == '44 a 60')
    & (fha['score_orig_r'] > 0)
    & (fha['x5001'] > 0)
]
print('FHA com DTI de 44 a 60%, score e reserva informados:', len(meio))

FHA com DTI de 44 a 60%, score e reserva informados: 1338


**O que se vê:** 1.338 pedidos têm DTI, score e reserva. Quase todos os que responderam sobre reserva também têm score.

**O que significa:** é esse número que entra na regra dos 400 casos, aplicada na célula seguinte.

### 2.4 Resultado da Checagem 2

A célula aplica as duas regras definidas no início da seção.

In [27]:
n_dti = fha.loc[fha['dti'] > 0, 'dti'].nunique()
n_score = fha.loc[fha['score_orig_r'] > 0, 'score_orig_r'].nunique()
c1 = n_dti >= 30 and n_score >= 100
c2 = len(meio) >= 400
print(f'1. regra dos valores exatos: DTI {n_dti}, score {n_score} -> '
      f'{"passa" if c1 else "não passa"}')
print(f'2. regra dos 400 casos: {len(meio)} -> '
      f'{"passa" if c2 else "não passa"}')

1. regra dos valores exatos: DTI 87, score 326 -> passa
2. regra dos 400 casos: 1338 -> passa


**Onde olhar:** as duas linhas impressas, cada uma com o veredito no fim.

**O que se vê:** as duas regras passam. Na faixa de 44% a 60% há 1.338 pedidos FHA com score e reserva, bem acima do mínimo de 400.

**O que significa:** a NSMO sustenta o segundo teste. Quais desses casos a análise manual aprovaria é visto na seção 7.2.

### 2.5 Os campos da NSMO medem o que a norma usa?

A Checagem 2 confirmou que DTI, score e reserva existem, e em quantidade. Falta ver se medem a mesma coisa que o Handbook, já que a matriz da análise manual vai ser aplicada sobre esses números. Dois pontos pedem análise, o score e o ano dos empréstimos, porque mudam o que entra na avaliação. As demais diferenças entre a NSMO e a norma não pedem ação agora e estão resumidas numa tabela no fim da seção.

**Pergunta:** a NSMO tem FHA aprovado com score que a norma não aceitaria?

In [28]:
score_valido = fha['score_orig_r'] > 0
abaixo_500 = fha[score_valido & (fha['score_orig_r'] < 500)]
print('FHA aprovados com score abaixo de 500:', len(abaixo_500))
colunas = ['open_year', 'dti', 'pti', 'ltv', 'score_orig_r', 'score_orig_s']
abaixo_500[colunas].sort_values('score_orig_r')

FHA aprovados com score abaixo de 500: 15


,open_year,dti,pti,ltv,score_orig_r,score_orig_s
61004,2024,59,16,50,463,-2
26340,2017,47,30,98,471,-2
54496,2023,47,45,98,473,-2
55768,2023,52,31,97,474,634
6781,2014,41,20,45,476,-2
43132,2020,53,25,98,476,744
34821,2019,50,21,99,478,574
60175,2024,26,26,17,478,-2
32100,2018,38,19,77,479,-2
59011,2024,37,36,93,488,728


**Onde olhar:** a coluna `score_orig_r` (score do respondente na contratação).

**O que se vê:** 15 FHA aprovados têm score entre 463 e 496. Foram contratados de 2014 a 2024, com DTI de 26% a 66%, sem concentração em um ano ou perfil.

**O que significa:** o FHA tem um score mínimo para qualquer aprovação. Se o score da NSMO fosse o mesmo que o banco usa, esses 15 casos não poderiam existir.

**Pergunta:** o que o Handbook diz sobre o score mínimo?

In [29]:
from src.handbook import excerpt, find_section

pag, texto = find_section('(3) Borrower Minimum Decision Credit Score')
print(f'Página {pag} do PDF\n')
print(excerpt(texto, '(3) Borrower Minimum', '(4) Borrower and Co-Borrower'))

Página 176 do PDF

(3) Borrower Minimum Decision Credit Score 
(a) Definition 
The Minimum Decision Credit Score (MDCS) refers to the credit score 
reported on the Borrower’s credit report when all reported scores are the same. 
Where three scores are reported, the median score is the MDCS. Where two 
differing scores are reported, the MDCS is the lowest score. Where only one 
score is reported, that score is the MDCS. 
An MDCS is determined for each Borrower. Where the Mortgage involves 
multiple Borrowers, the Mortgagee must determine the MDCS for each 
Borrower, and then select the lowest MDCS for all Borrowers.  
Where the Mortgage involves multiple Borrowers and one or more of the 
Borrowers do not have a credit score (non-traditional or insufficient credit), 
the Mortgagee must select the lowest MDCS of the Borrower(s) with credit 
score(s). 
(b) Eligibility Standard 
The Borrower is not eligible for FHA-insured financing if the MDCS is less 
than 500.


O trecho é a seção **II.A.1.b.ii(A)(3)**. O MDCS (Minimum Decision Credit Score, score mínimo de decisão) sai do relatório de crédito do banco. É a mediana quando há três scores e o menor quando há dois. Quando o pedido tem mais de um tomador, vale o menor MDCS entre eles. Abaixo de 500, ninguém é elegível ao FHA.

O codebook da NSMO (Appendix C, variável `Score_Orig_R`) define o campo como **VantageScore 3.0 na contratação, do respondente**. São duas diferenças em relação à norma:

- **O modelo.** O Handbook não nomeia o modelo de score, mas usa o do relatório de crédito do banco, que não precisa ser o VantageScore 3.0. Modelos diferentes dão números diferentes para a mesma pessoa, e não há tabela oficial de conversão. Os 15 aprovados abaixo de 500 indicam que as escalas não coincidem. O mais provável é que, no score que o banco usou, estivessem em 500 ou mais.
- **A pessoa.** A NSMO traz o score de quem respondeu a pesquisa; a norma usa o menor entre todos os tomadores. A célula seguinte mede o tamanho dessa diferença.

**Pergunta:** quando o pedido tem mais de um tomador, o respondente é quem tem o menor score?

In [30]:
# Score do cônjuge e de até três outros tomadores; -2 quando não há esse tomador
outros = ['score_orig_s', 'score_orig_o1', 'score_orig_o2', 'score_orig_o3']
todos_scores = fha[['score_orig_r'] + outros]
todos_scores = todos_scores.where(todos_scores > 0)  # códigos negativos viram vazio
fha['menor_score'] = todos_scores.min(axis=1)

tem_outro = todos_scores[outros].notna().any(axis=1)
outro_menor = tem_outro & (fha['menor_score'] < fha['score_orig_r'])
cruza_580 = (fha['score_orig_r'] >= 580) & (fha['menor_score'] < 580)
print('FHA com outro tomador com score:', tem_outro.sum())
print('  outro tomador com score menor que o do respondente:', outro_menor.sum())
print('  respondente com 580 ou mais, mas menor score abaixo de 580:',
      cruza_580.sum())

FHA com outro tomador com score: 3133
  outro tomador com score menor que o do respondente: 1538
  respondente com 580 ou mais, mas menor score abaixo de 580: 88


**Onde olhar:** a última linha, que conta quem muda de lado no corte de 580.

**O que se vê:** 3.133 FHA têm outro tomador com score. Em 1.538, esse outro tomador tem score menor que o respondente, e em 88 a troca leva o pedido de 580 ou mais para abaixo de 580.

**O que significa:** o score do respondente superestima o score que a norma manda usar em 1.538 dos 7.331 FHA. A diferença da pessoa tem correção, porque o menor score entre os tomadores fica mais perto do MDCS, e a escolha entre os dois é feita na preparação dos dados. A diferença do modelo não tem correção. O corte de 580 aplicado ao VantageScore é uma aproximação, e os achados que dependem dele (seção 7.2) são lidos com essa ressalva.

**Pergunta:** de quando são os empréstimos que responderam sobre reserva?

In [31]:
respondeu = (fha['x5001'] > 0).map({True: 'respondeu', False: 'sem resposta'})
pd.crosstab(fha['open_year'], respondeu, margins=True, margins_name='total')

x5001,respondeu,sem resposta,total
open_year,,,
2013,0,725,725
2014,0,694,694
2015,0,938,938
2016,0,815,815
2017,0,712,712
2018,0,578,578
2019,279,262,541
2020,368,0,368
2021,377,0,377


**Onde olhar:** a coluna "respondeu", ano a ano.

**O que se vê:** de 2013 a 2018, ninguém respondeu sobre reserva. Em 2019, pouco mais da metade respondeu, e de 2020 a 2024, todos.

**O que significa:** a pergunta sobre reserva só entrou na pesquisa em 2019, então o teste da NSMO usa empréstimos de 2019 a 2024. O agente, porém, vai julgá-los com a regra de hoje (Update 18, de 2026). Se a regra mudou nesse meio tempo, o agente pode discordar do banco por causa da mudança, e não por erro. A seção 5.4 compara as versões do Handbook e responde se mudou.

**O que mais a NSMO mede diferente da norma.** Os pontos abaixo não mudam quais casos entram na avaliação, mas mudam como ler o resultado da NSMO. Fontes: codebook da NSMO (Appendix C, v70) e documentação técnica da NMDB (National Mortgage Database), a base de onde a NSMO sai (*NMDB Technical Report 1*, FHFA, 2022, página 15 e tabela D-1).

| Campo | Diferença | Efeito no teste | O que se faz |
|---|---|---|---|
| Score | a norma usa o MDCS do relatório do banco, o menor entre os tomadores; a NSMO traz o VantageScore 3.0 do respondente | o corte de 580 fica aproximado | escolha do score na preparação dos dados; na avaliação, nota também sem os casos logo acima de 580 |
| Reserva | a norma pede reserva comprovada (seção 5.1); a NSMO traz a faixa declarada pelo cliente meses depois do fechamento (`x5001`) | a reserva parece maior do que era | ressalva no resultado |
| PTI | a norma conta a prestação completa, com imposto, seguros e condomínio (II.A.4); a NSMO não diz se inclui imposto e seguro | o PTI pode parecer menor do que era | na avaliação, teste com limite de PTI mais apertado |
| DTI | parte dos DTIs da NSMO foi estimada por modelo, sem marcação no arquivo (11,3% do DTI de parceiros como o FHA, de 2012 a 2019) | algumas discordâncias podem vir de número errado | ressalva no resultado |
| Ano | a NSMO cobre 2019 a 2024, e o agente usa a norma de 2026 | a regra poderia ter mudado | matriz e reserva iguais em todas as versões (seção 5.4) |

Score, reserva e PTI puxam para o mesmo lado. Na NSMO, o cliente parece um pouco melhor do que era. A nota da NSMO tende a sair otimista, e os dois testes da avaliação (sem os casos perto de 580 e com PTI mais apertado) medem quanto.

## 3. Resultado das checagens

| Checagem | Resultado | O que muda |
|---|---|---|
| 1. A negativa do HMDA aparece nos campos públicos? | regra dos 25% passa, regra do dobro não | a métrica passa a ser reportada por faixa de DTI |
| 2. A NSMO cobre a faixa do meio com score e reserva? | as duas regras passam | a NSMO assume o teste da faixa do meio, que a seção 5.2 divide depois em 50% |

**O projeto segue**, com a medição por faixa. As próximas seções exploram as duas bases e a norma a partir desse desenho.

## 4. Ausências nas duas bases

**Pergunta:** quanto de cada coluna não traz valor utilizável?

O HMDA é lido com todas as colunas como texto, porque os campos numéricos misturam números com marcadores como `NA` e `Exempt` (banco dispensado de informar).

In [32]:
def ausencias_hmda(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        'vazio (%)': df.isna().mean().mul(100).round(1),
        'Exempt (%)': df.eq('Exempt').mean().mul(100).round(1),
        'valores distintos': df.nunique(),
    })

print('HMDA:', hmda.shape)
ausencias_hmda(hmda)

HMDA: (2062934, 19)


,vazio (%),Exempt (%),valores distintos
lei,0.0,0.0,1612
state_code,1.2,0.0,54
action_taken,0.0,0.0,8
preapproval,0.0,0.0,2
loan_purpose,0.0,0.0,6
lien_status,0.0,0.0,2
loan_amount,0.0,0.0,653
loan_to_value_ratio,44.2,0.2,79132
property_value,21.7,0.2,587
occupancy_type,0.0,0.0,3


**Onde olhar:** a coluna "vazio (%)".

**O que se vê:** o DTI falta em 46,7% dos pedidos, o CLTV em 44,2%, a renda em 25,9% e o valor do imóvel em 21,7%. Os campos de decisão e finalidade vêm sempre preenchidos.

**O que significa:** quase metade do HMDA vem sem DTI, um número que assusta. Mas parte dos pedidos nem chega a ser analisada até o fim, então a ausência precisa ser aberta por decisão do banco.

**Pergunta:** em quais decisões o DTI, o CLTV e a renda faltam?

In [33]:
vazios = hmda[['debt_to_income_ratio', 'loan_to_value_ratio', 'income']].isna()
vazios['decisão'] = hmda['action_taken'].map(ACTION_TAKEN)
vazios.groupby('decisão').mean().mul(100).round(1)

,debt_to_income_ratio,loan_to_value_ratio,income
decisão,,,
"aprovado, não aceito",37.8,21.1,31.4
comprado de terceiro,100.0,100.0,90.5
fechado por incompletude,99.9,99.9,24.6
negado,13.7,9.2,8.3
originado,10.7,7.1,7.3
"pré aprovação aprovada, não aceita",4.2,5.2,0.3
pré aprovação negada,11.0,8.9,0.6
retirado pelo cliente,99.7,99.7,6.9


**Onde olhar:** as linhas "originado" e "negado", e depois as de comprado de terceiro, retirado pelo cliente e fechado por incompletude.

**O que se vê:** nos originados, o DTI falta em 10,7% e, nos negados, em 13,7%. Nos comprados de terceiro, retirados pelo cliente e fechados por incompletude, o DTI e o CLTV faltam em praticamente 100%.

**O que significa:** a ausência se concentra nos pedidos que não chegaram a uma análise completa. Nos dois grupos que servem de gabarito, ela é pequena.

**Pergunta:** e na NSMO?

Na NSMO, a ausência vem como código negativo, e não como vazio. O -2 quer dizer não se aplica (a pergunta não faz sentido para a pessoa), o -3 quer dizer não perguntado na onda (a pergunta ainda não existia quando ela respondeu) e o -4 quer dizer ausente (a pergunta valia, mas o valor não veio).

In [34]:
print('NSMO:', nsmo.shape, '| FHA:', fha.shape)
cols_nsmo = [c for c in nsmo.columns if c not in ('nsmoid',)]
pd.DataFrame({
    'tipo': fha[cols_nsmo].dtypes.astype(str),
    '-2 (%)': fha[cols_nsmo].eq(-2).mean().mul(100).round(1),
    '-3 (%)': fha[cols_nsmo].eq(-3).mean().mul(100).round(1),
    '-4 (%)': fha[cols_nsmo].eq(-4).mean().mul(100).round(1),
    'vazio (%)': fha[cols_nsmo].isna().mean().mul(100).round(1),
})

NSMO: (62359, 16) | FHA: (7331, 19)


,tipo,-2 (%),-3 (%),-4 (%),vazio (%)
survey_wave,int64,0.0,0.0,0.0,0.0
x36c,int64,34.9,0.0,0.0,0.0
x36d,int64,28.7,19.0,0.0,0.0
x5001,int64,0.0,64.4,0.0,0.0
open_year,int64,0.0,0.0,0.0,0.0
ltv,int64,0.0,0.0,0.0,0.0
cltv,int64,0.0,0.0,0.0,0.0
dti,int64,0.0,0.0,0.0,0.0
pti,int64,0.0,0.0,0.1,0.0
loan_type,int64,0.0,0.0,0.0,0.0


**Onde olhar:** as linhas `x5001` (reserva), `score_orig_r`, `x36c` e `x36d`.

**O que se vê:** nenhuma coluna tem vazio. A reserva não foi perguntada em 64,4% dos FHA, por causa das ondas, e o score falta em só 0,7%. `x36c` e `x36d`, usadas na seção 6, só valem para compras, e `x36d` não foi perguntada em 19,0% dos FHA.

**O que significa:** na NSMO, a única ausência que pesa é a da reserva, já medida nas seções 2.3 e 2.5. As outras colunas chegam quase completas.

### 4.1 Quem fica fora da comparação com o banco, e por quê

A ideia inicial era comparar a recomendação de suspender com os pedidos fechados por incompletude. A tabela por decisão mostra que esses pedidos chegam sem DTI e sem CLTV em 99,9% dos casos. Para um pedido assim, suspender por falta de dado é a única resposta possível, e qualquer regra do tipo "se está vazio, suspende" acertaria todos. A concordância nessa classe inflaria a nota do agente sem medir nada sobre ele.

**Decisão:** os fechados por incompletude saem da comparação com a decisão do banco, como os retirados pelo cliente. A ação de suspender continua existindo e é medida onde tem mérito, nos pedidos com DTI acima de 43%. Eles vêm preenchidos, e o agente precisa perceber que faltam score e reserva. Até onde essa faixa vai é decidido na seção 5.2, depois da leitura da norma.

A remoção é aplicada na preparação dos dados (notebook 02), junto das demais exclusões, cada uma precedida da sua evidência. São 102.018 pedidos fechados por incompletude (seção 1.1).

## 5. Leitura do Handbook 4000.1

Os limites do resumo da abertura vêm daqui. A seção os confere no texto oficial do HUD Handbook 4000.1 (Update 18, publicada em 12 de agosto de 2026), baixado do site do HUD. O texto é extraído direto do PDF, sem edição.

### 5.1 Limites da análise manual no texto da norma

**Pergunta:** o que o texto oficial diz sobre os limites de DTI e PTI na análise manual?

In [35]:
pag, texto = find_section('viii. Approvable Ratio Requirements (Manual)')
print(f'Página {pag} do PDF\n')
print(excerpt(texto, 'viii. Approvable Ratio', 'ix. Documenting'))

Página 384 do PDF

viii. Approvable Ratio Requirements (Manual) 
The maximum Total Mortgage Payment to Effective Income Ratio (PTI) and Total Fixed 
Payments to Effective Income Ratio, or DTI, applicable to manually underwritten 
Mortgages are summarized in the matrix below. 
The qualifying ratios for Borrowers with no credit score are computed using income only 
from Borrowers occupying the Property and obligated on the Mortgage. Non-occupant 
co-Borrower income may not be included.  
Lowest 
Minimum 
Decision 
Credit Score 
Maximum 
Qualifying 
Ratios (%) 
Acceptable Compensating Factors 
500-579 or No 
Credit Score 
31/43 Not applicable. Borrowers with Minimum Decision Credit 
Scores below 580, or with no credit score may not exceed 
31/43 ratios. 
 
Energy Efficient Homes may have stretch ratios of 33/45. 
580 and above 31/43 No compensating factors required. 
 
Energy Efficient Homes may have stretch ratios of 33/45. 
580 and above 37/47 One of the following: 
• verified and docum

O trecho está na seção **II.A.5.d.viii** (Approvable Ratio Requirements, Manual). Cada linha da matriz traz dois limites. O primeiro é o PTI (só a prestação do imóvel sobre a renda) e o segundo é o DTI (todas as dívidas sobre a renda). Lida como tabela:

| Score mínimo | PTI/DTI máximo | Exigência |
|---|---|---|
| 500 a 579, ou sem score | 31/43 | não pode passar disso |
| 580 ou mais | 31/43 | nenhuma |
| 580 ou mais | 37/47 | um fator compensatório |
| 580 ou mais | 40/40 | nenhuma dívida além da moradia |
| 580 ou mais | 40/50 | dois fatores compensatórios |

Os fatores compensatórios estão descritos na seção seguinte (II.A.5.d.ix). São a reserva de pelo menos três prestações para imóveis de uma ou duas unidades, o aumento pequeno da prestação em relação ao gasto atual com moradia, a renda residual pela tabela do VA e a renda extra não computada, esta última só em conjunto com outro fator. A matriz não tem nenhuma linha acima de 50% de DTI.

A reserva é o único desses fatores que a NSMO permite verificar. O trecho dela vem a seguir.

In [36]:
pag, texto = find_section('(B) Verified and Documented Cash Reserves')
print(f'Página {pag} do PDF\n')
print(excerpt(texto, '(B) Verified and Documented', '(C) Minimal Increase'))

Página 385 do PDF

(B) Verified and Documented Cash Reserves 
Verified and documented cash Reserves may be cited as a compensating factor 
subject to the following requirements. 
• Reserves are equal to or exceed three total monthly Mortgage Payments (one 
and two units); or 
• Reserves are equal to or exceed six total monthly Mortgage Payments (three 
and four units). 
Reserves are calculated as the Borrower’s total assets as described in Asset 
Requirements (Manual) less: 
• the total funds required to close the Mortgage; 
• Gifts; 
• borrowed funds; and 
• cash received at closing in a cash-out refinance transaction or incidental cash 
received at closing in the mortgage transaction.


O trecho é a seção **II.A.5.d.ix(B)**. A reserva vale como fator quando cobre pelo menos três prestações completas (imóvel de uma ou duas unidades), com comprovação. Ela é calculada como o total de ativos menos o dinheiro do fechamento, os presentes, os empréstimos e o dinheiro recebido num refinanciamento com saque. A pergunta da NSMO (`x5001`) é sobre a mesma ideia, o que sobrou depois do fechamento contado em prestações, e a faixa "3 a 6 meses" começa exatamente no mínimo da norma. As diferenças entre as duas estão na tabela do fim da seção 2.5.

### 5.2 O teto de 50% contra o dado

**Pergunta:** se a análise manual para em 50%, quantos aprovados da NSMO estão acima disso?

Um pedido com DTI de 55% só pode ter sido aprovado pelo sistema automático do FHA, que aceita DTI mais alto quando o resto do perfil compensa. O HMDA publica a faixa de 50% a 60% agrupada, com o 50% exato dentro dela; a NSMO traz o valor exato e permite separar o 50% do restante.

In [37]:
fha['faixa_manual'] = pd.cut(
    fha['dti'], bins=[0, 43, 50, 60, 100],
    labels=['até 43', '44 a 50', '51 a 60', 'acima de 60'],
)
completos = (fha['score_orig_r'] > 0) & (fha['x5001'] > 0)
pd.DataFrame({
    'FHA aprovados': fha['faixa_manual'].value_counts(),
    'com score e reserva': fha.loc[completos, 'faixa_manual'].value_counts(),
}).sort_index()

,FHA aprovados,com score e reserva
faixa_manual,,
até 43,4018,1243
44 a 50,1870,718
51 a 60,1319,620
acima de 60,124,7


**Onde olhar:** a linha "51 a 60".

**O que se vê:** 1.319 FHA aprovados têm DTI de 51% a 60%, faixa que a matriz manual não permite. Na faixa de 44% a 50%, 718 têm score e reserva.

**O que significa:** esses 1.319 pedidos passaram pelo sistema automático. Um agente que aplica corretamente a regra manual responderia "não aprovável pela análise manual" e seria contado como erro por uma decisão que não foi de um analista.

Abaixo de 50%, não dá para saber por qual via o pedido foi aprovado. A NSMO não diz se o empréstimo passou pelo sistema automático ou por um analista. Se o sistema automático aprova casos acima de 50%, ele também aprova casos abaixo, e um aprovado com DTI de 46% pode nunca ter passado por um analista. Na NSMO, "aprovado" quer dizer que o caso era aprovável por alguma via, e não que a análise manual o aprovaria (seção 7.2).

**Decisão: a faixa do meio é dividida no teto de 50%.**

- De 44% a 50%, o agente é avaliado. No HMDA (44% a 49%, pelo agrupamento publicado), espera-se que ele suspenda pedindo score e reserva. Na NSMO, a forma de avaliar depende do que a base mostra (seção 7.2).
- De 51% a 60% (no HMDA, a faixa agrupada de 50% a 60%), os casos saem da nota e são reportados à parte, como aprovações do sistema automático.

A Checagem 2 continua válida com o corte. Dos 1.338 pedidos com score e reserva entre 44% e 60%, 718 ficam na faixa avaliada, acima do mínimo de 400, e os outros 620 vão para o relatório à parte.

No HMDA, a célula abaixo conta quantos pedidos da faixa de 50% a 60% saem da nota, por decisão do banco.

In [38]:
# HMDA: pedidos da faixa de 50% a 60% que saem da nota, por decisão
faixa_50_60 = hmda[dti_band(hmda['debt_to_income_ratio']) == '50 a 60']
print('HMDA, DTI de 50% a 60%:', len(faixa_50_60))
faixa_50_60['action_taken'].map(ACTION_TAKEN).value_counts().to_frame('pedidos')

HMDA, DTI de 50% a 60%: 319484


,pedidos
action_taken,
originado,243086
negado,59476
"aprovado, não aceito",7927
"pré aprovação aprovada, não aceita",6298
pré aprovação negada,2697


**Onde olhar:** o total e as duas primeiras linhas.

**O que se vê:** a faixa de 50% a 60% tem 319.484 pedidos no HMDA, entre eles 243.086 originados e 59.476 negados. As pré aprovações já saem pelo motivo da seção 1.1.

**O que significa:** os 243.086 originados, os 59.476 negados e os 7.927 aprovados, não aceitos dessa faixa vão para o relatório à parte. O teste no HMDA fica com as faixas até 49% e acima de 60%.

### 5.3 Teto de CLTV no refinanciamento com saque

**Pergunta:** qual é o teto de CLTV do refinanciamento com saque no texto oficial?

In [39]:
pag, texto = find_section('(a) Maximum Loan-to-Value')
print(f'Página {pag} do PDF\n')
print(excerpt(texto, '(B) Maximum Mortgage Amounts', '(c) Nationwide'))

Página 463 do PDF

(B) Maximum Mortgage Amounts 
(1) Standard 
(a) Maximum Loan-to-Value 
The maximum LTV is 80 percent of the Adjusted Value. 
(b) Maximum Combined Loan-to-Value 
The maximum CLTV is 80 percent of the Adjusted Value.


Para refinanciamento com saque (seção II.A.8), o teto de LTV e de CLTV é 80% do valor ajustado do imóvel, bem abaixo dos 96,5% da compra. É o teto que a seção 1.5 encontrou no dado. Passar de 80% é raro entre os refinanciamentos com saque aprovados e comum entre os negados por garantia.

### 5.4 Qual versão da norma vale

Na Update 18, as seções de análise manual (II.A.5.d) e de refinanciamento (II.A.8) trazem no título a data 11/10/2026, que no formato americano (mês e dia) quer dizer 10 de novembro de 2026. Na Update 17, a seção II.A.5.d traz 08/19/2024, ou 19 de agosto de 2024. Se a data marcar a entrada em vigor, a versão publicada em agosto ainda não seria a única válida hoje.

**Pergunta:** o que o documento de abertura da Update 18 diz sobre essas datas?

In [40]:
from src.handbook import HANDBOOK, changed_sentences, pages, section_text

abertura = '\n'.join(pages('update_18')[:9])
print(excerpt(abertura, '3. Implementation:', '4. Public Feedback'))
print('...')
print(excerpt(abertura, '2. Effective dates are shown', '3. Yellow'))

3. Implementation: 
Below is a list of effective dates for the Handbook changes: 
Handbook changes identified in Section 2 of this 
Transmittal incorporated to reflect a Mortgagee 
Letter 
Effective as previously announced in the 
corresponding Mortgagee Letter. 
All other changes  May be implemented immediately, but must be 
implemented no later than November 10, 2026.
...
2. Effective dates are shown at the end of heading titles, at the 4th level (e.g., I.A.1.a) in 
parentheses.


A data entre parênteses é a de vigência. As mudanças da Update 18 que não vêm de Mortgagee Letter podem ser aplicadas desde a publicação e passam a ser obrigatórias em 10 de novembro de 2026. Até lá valem as duas versões, e depois só a 18.

**Pergunta:** a escolha da versão muda alguma regra usada pelo projeto? O texto das seções que o agente aplica é comparado frase a frase entre as duas versões.

In [41]:
trechos = {
    'matriz e fatores (II.A.5.d.viii e ix)': (
        'viii. Approvable Ratio Requirements (Manual)',
        '(1) Calculating Residual Income',
    ),
    'refinanciamento com saque (II.A.8)': (
        'Cash-out refinance transactions are only permitted',
        '(c) Nationwide Mortgage Limit',
    ),
}
for nome, (ini, fim) in trechos.items():
    diferentes = changed_sentences('update_17', 'update_18', ini, fim)
    print(f'== {nome}')
    if diferentes:
        print('\n'.join(diferentes))
    else:
        print('sem diferença de texto')
    print()

== matriz e fatores (II.A.5.d.viii e ix)
sem diferença de texto

== refinanciamento com saque (II.A.8)
-(b) required documentation if the mortgage on the subject property is not reported in the borrower’s credit report or is not in the name of the borrower, the mortgagee must obtain a verification of mortgage, bank statements or other documentation to evidence that all payments have been made by the borrower in the month due for the previous 12 months.
+(b) required documentation [text was deleted in this section.] if the mortgage on the subject property is not reported in the borrower’s credit report or is not in the name of the borrower, the mortgagee must obtain a verification of mortgage, bank statements or other documentation to evidence that all payments have been made by the borrower in the month due for the previous 12 months.
-a forbearance plan is not required if the forbearance was due to the impacts of the covid-19 national emergency.



**Onde olhar:** as linhas que começam com `-` (só na Update 17) e `+` (só na Update 18).

**O que se vê:** na matriz e nos fatores compensatórios, não há diferença de texto. No refinanciamento com saque, o teto de 80% é o mesmo. Saiu apenas a dispensa do plano de pausa de pagamento (forbearance) concedida por causa da COVID-19, e a Update 18 marca o corte com "[text was deleted in this section.]".

**O que significa:** as regras testadas no projeto são idênticas nas duas versões.

**Decisão: o índice da norma usa a Update 18.** É a versão que já pode ser aplicada e será a única válida a partir de 10 de novembro de 2026, e a escolha não altera a avaliação. A Update 17 fica guardada como candidata à demonstração de mudança de norma. Como cada seção tem sua própria data de vigência, essa data é candidata a metadado de cada trecho no índice.

**Pergunta:** e para os empréstimos da NSMO? A reserva só existe de 2019 a 2024 (seção 2.5), e o agente vai julgá-los com a Update 18. A matriz e a reserva mínima eram as mesmas nesse período? Para responder, foram usadas as versões antigas do Handbook guardadas no arquivo do HUD, que são todas as de 2019, a de 2020 e a de dezembro de 2016, que valeu até março de 2019 (o arquivo não tem 2017 nem 2018). Primeiro, a célula procura em cada versão as regras que o agente aplica, que são o corte de score 580, os quatro limites da matriz e a reserva de três prestações.

In [42]:
INICIO = 'viii. Approvable Ratio Requirements (Manual)'
FIM = '(1) Calculating Residual Income'
REGRAS = {
    'score 580': '580 and above',
    '31/43': '31/43',
    '37/47': '37/47',
    '40/40': '40/40',
    '40/50': '40/50',
    'reserva de 3 prestações': 'three total monthly mortgage payments',
}

linhas = []
for versao in HANDBOOK:
    texto = section_text(versao, INICIO, FIM)
    linha = {'versão': versao}
    for nome, frase in REGRAS.items():
        if frase in texto:
            linha[nome] = 'sim'
        else:
            linha[nome] = 'não'
    linhas.append(linha)
pd.DataFrame(linhas).set_index('versão')

,score 580,31/43,37/47,40/40,40/50,reserva de 3 prestações
versão,,,,,,
2016_12,sim,sim,sim,sim,sim,sim
2019_03,sim,sim,sim,sim,sim,sim
2019_07,sim,sim,sim,sim,sim,sim
2019_08,sim,sim,sim,sim,sim,sim
2019_10,sim,sim,sim,sim,sim,sim
2019_12,sim,sim,sim,sim,sim,sim
2020_11,sim,sim,sim,sim,sim,sim
update_17,sim,sim,sim,sim,sim,sim
update_18,sim,sim,sim,sim,sim,sim


**Onde olhar:** se alguma célula da tabela diz "não".

**O que se vê:** todas dizem "sim". As nove versões, de 2016 a 2026, trazem o corte de 580, os quatro limites e a reserva de três prestações.

**O que significa:** as regras centrais estão no texto de todas as versões. Essa checagem só acha o que se sabe procurar, então a célula seguinte compara o trecho inteiro da matriz e dos fatores, frase a frase, entre cada versão antiga e a Update 18, com a mesma função usada acima.

In [43]:
for versao in HANDBOOK:
    if versao == 'update_18':
        continue
    diferentes = changed_sentences(versao, 'update_18', INICIO, FIM)
    print(f'{versao}: {len(diferentes)} frases diferentes da Update 18')

print('\nDiferenças da versão de março de 2019 (- antiga, + Update 18):')
for linha in changed_sentences('2019_03', 'update_18', INICIO, FIM):
    print('  ', linha[:110])

2016_12: 16 frases diferentes da Update 18
2019_03: 13 frases diferentes da Update 18
2019_07: 13 frases diferentes da Update 18
2019_08: 13 frases diferentes da Update 18
2019_10: 13 frases diferentes da Update 18
2019_12: 13 frases diferentes da Update 18
2020_11: 13 frases diferentes da Update 18
update_17: 0 frases diferentes da Update 18

Diferenças da versão de março de 2019 (- antiga, + Update 18):
   -or • the applicable international energy conservation code (iecc) year used by the state or local building co
   +• the applicable international energy conservation code (iecc) year used by the state or local building code;
   +or • a manufactured home that is certified as energy star to their quality assurance provider and ensure that
   -or • homes where documented cost-effective energy improvements, as identified in the home energy score report
   +• homes where documented cost-effective energy improvements, as identified in the home energy score report, w
   +or • a manufactur

**Onde olhar:** a contagem por versão e, na lista, o assunto de cada frase.

**O que se vê:** as versões de 2019 e 2020 diferem da Update 18 nas mesmas 13 frases, e a de 2016 em 16. Quase todas falam de casas com eficiência energética, porque as versões novas passaram a aceitar casa pré-fabricada com selo ENERGY STAR. A Update 17 não tem diferença.

**O que significa:** nenhuma das bases diz se o imóvel tem selo, então a regra das casas eficientes fica fora do teste. A matriz e a reserva mínima não mudaram de 2016 a 2026. Se o agente discordar de uma aprovação da NSMO, a causa não é mudança de regra. No HMDA a questão não existe, porque os pedidos são todos de 2025.

## 6. Campos que a norma cita

Além do DTI e do CLTV (seção 1.5), a norma muda conforme a finalidade do pedido. Por isso a finalidade entra no perfil do caso que o agente recebe, qualquer que seja o dado, porque sem ela o agente não sabe qual teto de CLTV aplicar. A seção mostra como a finalidade se distribui por decisão do banco, para decidir como sortear os grupos de avaliação, e depois volta à compra com CLTV acima do teto.

**Pergunta:** a finalidade do pedido muda de uma decisão para outra? Se mudar, o sorteio do grupo de ajuste e do grupo da nota final, no notebook 02, mantém a proporção de cada finalidade, porque um grupo final de algumas centenas de casos poderia sair com mais pedidos de uma finalidade só por acaso, e a nota mudaria junto. Se não mudar, um sorteio simples basta.

In [44]:
decididos = hmda[hmda['action_taken'].isin(['1', '2', '3', '5'])].copy()
decididos['decisão'] = decididos['action_taken'].map(ACTION_TAKEN)
finalidade = decididos['loan_purpose'].map(LOAN_PURPOSE)
pd.crosstab(finalidade, decididos['decisão'], normalize='columns').mul(100).round(1)

decisão,"aprovado, não aceito",fechado por incompletude,negado,originado
loan_purpose,,,,
compra,39.3,12.2,36.2,70.6
não se aplica,0.1,0.0,0.2,0.0
outro,0.9,4.9,3.4,1.2
refinanciamento com saque,18.9,44.3,41.1,15.2
refinanciamento sem saque,40.5,37.2,18.5,12.6
reforma,0.2,1.4,0.7,0.3


**Onde olhar:** as colunas "negado" e "originado". Cada coluna soma 100%.

**O que se vê:** o refinanciamento com saque é 41,1% dos negados e 15,2% dos originados. A compra é 36,2% contra 70,6%.

**O que significa:** a finalidade muda muito a chance de negativa. Um grupo final com refinanciamentos com saque a mais teria mais negados, e a nota do agente mudaria por causa do sorteio, não do raciocínio.

**Decisão: sorteio estratificado por finalidade.** No notebook 02, o grupo de ajuste e o grupo da nota final são sorteados mantendo a proporção de cada finalidade.

**Pergunta (H3):** a NSMO confirma que o segundo financiamento explica as compras com CLTV acima do teto?

A seção 1.5 mostrou que metade das compras aprovadas acima de 96,5% passa de 100%, o que a norma só permite com segundo financiamento de órgão público ou entidade aprovada, e o HMDA não publica esse dado. A NSMO tem duas perguntas sobre a origem do dinheiro, feitas só a quem comprou o imóvel. A `x36c` pergunta se o cliente usou ajuda ou empréstimo de órgão público ou entidade sem fins lucrativos, e a `x36d` pergunta se usou um segundo financiamento.

**H3: o segundo financiamento explica as compras acima do teto?** Se explicar, a parcela de compras com alguma das duas respostas cresce junto com o CLTV e fica alta acima de 100%. Se não tivesse relação, a parcela seria parecida em todas as faixas de CLTV. A NSMO publica o CLTV como número inteiro, então a faixa de 98 a 100 junta casos com e sem seguro financiado.

In [45]:
from src.nsmo import X36

# Só compras FHA com CLTV informado e resposta sobre a origem do dinheiro
compras = fha[fha['x36c'].isin(X36.keys()) & (fha['cltv'] > 0)].copy()
compras['faixa_cltv'] = pd.cut(
    compras['cltv'], bins=[0, 96, 97, 100, 200],
    labels=['até 96', '97', '98 a 100', 'acima de 100'],
)
compras['ajuda pública'] = compras['x36c'] == 1
compras['segundo financiamento'] = compras['x36d'] == 1
compras['alguma das duas'] = (
    compras['ajuda pública'] | compras['segundo financiamento']
)
origem = compras.groupby('faixa_cltv', observed=True)[
    ['ajuda pública', 'segundo financiamento', 'alguma das duas']
].mean().mul(100).round(1)
origem.insert(0, 'compras', compras.groupby('faixa_cltv', observed=True).size())
display(origem)

dentro = origem.loc['até 96', 'alguma das duas']
acima = origem.loc['acima de 100', 'alguma das duas']
print(f'acima de 100 contra até 96: {acima / dentro:.1f} vezes')
print(f'acima de 100 sem nenhuma das duas: {100 - acima:.1f}%')

,compras,ajuda pública,segundo financiamento,alguma das duas
faixa_cltv,,,,
até 96,3303,10.3,1.3,11.1
97,481,12.1,1.0,12.7
98 a 100,838,18.3,2.7,19.8
acima de 100,153,31.4,9.8,39.2


acima de 100 contra até 96: 3.5 vezes
acima de 100 sem nenhuma das duas: 60.8%


**Onde olhar:** a coluna "alguma das duas", de cima para baixo, a coluna "compras" na última linha e as duas contas impressas abaixo da tabela.

**O que se vê:** até 96% de CLTV, 11,1% das compras usaram ajuda pública ou segundo financiamento. A parcela sobe a cada faixa, com 12,7% em 97, 19,8% de 98 a 100 e 39,2% acima de 100. A faixa acima de 100 tem só 153 compras.

**Resposta à H3.** Em parte. O mecanismo que a norma permite existe e cresce com o CLTV. Acima de 100%, a parcela é cerca de três vezes e meia a das compras dentro do teto. Mas ele explica só uma parte, porque 6 em cada 10 compras acima de 100% não declararam nenhuma das duas. Pode ser esquecimento do cliente, ajuda dada como doação (que não vira dívida), empréstimo de parente (a norma também permite) ou outra causa. Há três ressalvas. São poucos casos, a resposta vem do cliente depois do fechamento, e o `x36d` não foi perguntado nas primeiras ondas da pesquisa, o que puxa a coluna do segundo financiamento para baixo.

**Premissa para o agente:** numa compra, CLTV acima de 96,5% não é, sozinho, violação da norma. O agente trata o caso como condição a confirmar. De 96,5% a 98,2%, confirma se a diferença é o seguro financiado; acima de 98,2%, se há segundo financiamento de órgão público ou entidade aprovada (II.A.5). Supor que todo cliente acima do teto recebeu ajuda seria errado em cerca de 6 de cada 10 casos e levaria o agente a aprovar compras que o banco negou por garantia, o erro mais caro do projeto. A forma de pontuar essa resposta é fixada na seção 7.2, junto com o caso dos fatores compensatórios.

Na NSMO, `x36c` e `x36d` dizem, cliente a cliente, se houve ajuda. O banco conhece essa informação na hora de decidir, porque o segundo financiamento é declarado no pedido. Se ela entra como dado do caso entregue ao agente é decidido na preparação dos dados.

## 7. Prévia da norma sobre o dado

**A pergunta.** Para cada pedido na faixa avaliada, qual linha da matriz manual (seção 5.1) ele aciona, e a base traz a informação para verificar a exigência? A seção não calcula acerto, nem de regra nem de agente. O grupo da nota final ainda não foi separado, e medir acerto na base inteira agora anteciparia o resultado antes de a meta ser fixada.

A matriz foi escrita em código em `src/rules.py`, a partir do texto da seção 5.1, e será a base do teto de referência da avaliação. O teto de referência é a nota que a norma inteira, aplicada em código e sem IA, tiraria nos mesmos casos. Ele mostra o máximo que se pode esperar de um agente que segue a norma com a informação da base.

### 7.1 HMDA: a matriz pede score, e o HMDA não publica score

**Pergunta:** o HMDA traz algum campo com o valor do score?

In [46]:
print('Campos de score no HMDA carregado:',
      [c for c in hmda.columns if 'score' in c])
tipos = hmda['applicant_credit_score_type'].value_counts().head(6).to_frame('pedidos')
tipos['%'] = (tipos['pedidos'] / len(hmda) * 100).round(1)
tipos

Campos de score no HMDA carregado: ['applicant_credit_score_type']


,pedidos,%
applicant_credit_score_type,,
9,1007807,48.9
1,360732,17.5
2,337091,16.3
3,321957,15.6
7,25923,1.3
8,4327,0.2


**Onde olhar:** a lista de campos e a contagem por código.

**O que se vê:** o único campo de score é `applicant_credit_score_type`, que diz só qual modelo de score o banco usou (códigos do CFPB, como Equifax Beacon ou FICO Risk Score). O código 9, "não se aplica", cobre 48,9% das linhas.

**O que significa:** o HMDA nunca traz o valor do score. Na faixa de 44% a 49%, toda linha da matriz depende do score (a partir de 580) e de fator compensatório, e nenhum dos dois pode ser verificado. Para esses pedidos, a norma só permite aprovar com uma condição que a base não mostra, e a resposta esperada do agente é suspender pedindo score e reserva, como definido nas seções 1.7 e 5.2. Acima de 50%, nenhuma linha permite a aprovação manual.

Até 43%, a matriz aprova sem fator para qualquer score a partir de 500, desde que o PTI fique em até 31%. O HMDA não publica o PTI. A célula a seguir mede, na NSMO, quantos pedidos FHA até 43% passariam desse limite.

In [47]:
# NSMO: FHA com DTI até 43% e PTI informado
ate_43 = fha[(fha['dti'] > 0) & (fha['dti'] <= 43) & (fha['pti'] > 0)].copy()
ate_43['faixa de DTI'] = pd.cut(
    ate_43['dti'], bins=[0, 35, 40, 43],
    labels=['até 35%', '36% a 40%', '41% a 43%'],
)
ate_43['PTI acima de 31%'] = ate_43['pti'] > 31

grupos = ate_43.groupby('faixa de DTI', observed=True)['PTI acima de 31%']
tabela = grupos.agg(['size', 'sum'])
tabela.columns = ['pedidos', 'PTI acima de 31%']
tabela['%'] = (tabela['PTI acima de 31%'] / tabela['pedidos'] * 100).round(1)
print('FHA da NSMO com DTI até 43% e PTI informado:', len(ate_43))
print('  desses, com PTI acima de 31%:', ate_43['PTI acima de 31%'].sum())
tabela

FHA da NSMO com DTI até 43% e PTI informado: 4014
  desses, com PTI acima de 31%: 509


,pedidos,PTI acima de 31%,%
faixa de DTI,,,
até 35%,1978,64,3.2
36% a 40%,1197,236,19.7
41% a 43%,839,209,24.9


**Onde olhar:** a coluna de percentual, faixa a faixa.

**O que se vê:** até 35% de DTI, só 3,2% dos pedidos passam de 31% de PTI. De 36% a 40% são 19,7%, e de 41% a 43%, 24,9%. No total, 509 de 4.014.

**O que significa:** perto do teto de 43%, cerca de 1 pedido em cada 5 precisaria, pela norma, de um fator compensatório que o DTI sozinho não mostra. Como o PTI da NSMO pode estar subestimado (seção 2.5), esses percentuais são um piso. A falta do PTI no HMDA, porém, é uma lacuna da base e não uma pendência do cliente. O banco calcula o PTI com dados que já tem, e nenhum analista suspenderia um pedido para pedir esse número.

**Decisão: PTI ausente no HMDA.** Até 43%, a resposta esperada continua sendo aprovar, com o PTI marcado como não informado no memorando. A concordância dessa faixa sai em duas partes, até 35% e de 36% a 43%, para mostrar se a falta do PTI pesa no resultado.

### 7.2 NSMO: o que a norma exige na faixa de 44% a 50%

**Pergunta:** com DTI, PTI e score exatos, quantos pedidos têm a exigência da norma verificável?

Das exigências da matriz, só a reserva pode ser verificada na NSMO. Renda residual, aumento pequeno da prestação, renda extra e ausência de dívida discricionária não estão na base. A reserva da NSMO é declarada pelo cliente e é uma aproximação do fator da norma (seção 2.5). As categorias "3 a 6 meses" e "7 meses ou mais" atendem ao mínimo de três prestações para imóveis de uma ou duas unidades.

In [48]:
from src.rules import manual_requirements, norm_reading

caso = fha[completos & (fha['faixa_manual'] == '44 a 50')].copy()
caso['pti_num'] = caso['pti'].where(caso['pti'] > 0)
caso['exigência'] = manual_requirements(
    caso['dti'], caso['pti_num'], caso['score_orig_r']
)
caso['reserva de 3+ prestações'] = caso['x5001'].isin([3, 4]).map(
    {True: 'sim', False: 'não'}
)
print('FHA com DTI de 44% a 50%, score e reserva:', len(caso))
pd.crosstab(caso['exigência'], caso['reserva de 3+ prestações'],
            margins=True, margins_name='total')

FHA com DTI de 44% a 50%, score e reserva: 718


reserva de 3+ prestações,não,sim,total
exigência,,,
1 fator,195,99,294
2 fatores,161,121,282
fora da matriz,78,62,140
inelegível (score abaixo de 500),2,0,2
total,436,282,718


**Onde olhar:** as linhas de exigência e a coluna "sim" (reserva de três prestações ou mais).

**O que se vê:** 294 pedidos precisavam de um fator, 282 de dois, e 140 estão fora da matriz. Outros 2 têm score abaixo de 500 e nem entram no FHA. Em cada linha, a maioria não tem a reserva de três prestações.

**O que significa:** a reserva sozinha não fecha a exigência da maioria dos casos. A célula seguinte resume, pedido a pedido, o que a norma permite concluir com o que a base traz.

In [49]:
# norm_reading (src/rules.py) cruza a exigência com a reserva de cada pedido
tem_reserva = caso['reserva de 3+ prestações'] == 'sim'
leituras = []
for exigencia, reserva in zip(caso['exigência'], tem_reserva):
    leituras.append(norm_reading(exigencia, reserva))
caso['leitura da norma'] = leituras
prev = caso['leitura da norma'].value_counts().to_frame('pedidos')
prev['%'] = (prev['pedidos'] / len(caso) * 100).round(1)
display(prev)

# Pedidos em que falta informação para fechar a exigência
verificada = prev.loc['exigência verificada', 'pedidos']
fora_matriz = prev.loc['fora da matriz manual', 'pedidos']
falta_info = len(caso) - verificada - fora_matriz
print(f'falta informação: {falta_info} ({falta_info / len(caso):.1%})')

,pedidos,%
leitura da norma,,
fator não verificado,195,27.2
nenhum de 2 fatores verificado,161,22.4
fora da matriz manual,142,19.8
1 de 2 fatores verificado,121,16.9
exigência verificada,99,13.8


falta informação: 477 (66.4%)


**Onde olhar:** a linha "exigência verificada", as que falam de fator não verificado e a contagem abaixo da tabela.

**O que se vê:** dos 718 pedidos, só 99 (13,8%) têm a exigência inteiramente verificável, porque precisavam de um fator e têm reserva de três prestações ou mais. Em 477 (66,4%) falta informação para fechar a exigência. O fator pode existir (renda residual, aumento pequeno da prestação, renda extra), mas a base não mostra. Os 142 restantes (19,8%) estão fora da matriz manual, por score abaixo de 580 ou PTI acima de 40%.

**O que significa:** para a maioria dos casos, a leitura fiel da norma não é "aprovar", e sim "aprovar se o fator X for comprovado", ou suspender pedindo a comprovação. Contar essa resposta como erro puniria o agente por seguir a norma com a informação que tem, o mesmo problema que levou à separação da faixa de 51% a 60%. Mesmo dentro da matriz, a aprovação na NSMO não garante que um analista conferiu os fatores, porque a base não diz por qual via o empréstimo passou (seção 5.2).

**Pergunta:** os 142 fora da matriz saem por qual limite? A resposta importa porque o score da NSMO não é o da norma (seção 2.5).

In [50]:
fora = caso[caso['leitura da norma'] == 'fora da matriz manual']
score_baixo = fora['score_orig_r'] < 580
score_500 = fora['score_orig_r'] < 500
pti_alto = (fora['score_orig_r'] >= 580) & (fora['pti'] > 40)
print('fora da matriz:', len(fora))
print('  score abaixo de 580 (escala da NSMO):', score_baixo.sum())
print('    dos quais abaixo de 500, inelegíveis ao FHA:', score_500.sum())
print('  score de 580 ou mais e PTI acima de 40%:', pti_alto.sum())

# Mesma matriz, com o menor score entre os tomadores (mais perto do MDCS)
caso['exigência pelo menor score'] = manual_requirements(
    caso['dti'], caso['pti_num'], caso['menor_score']
)
pd.crosstab(caso['exigência'], caso['exigência pelo menor score'])

fora da matriz: 142
  score abaixo de 580 (escala da NSMO): 35
    dos quais abaixo de 500, inelegíveis ao FHA: 2
  score de 580 ou mais e PTI acima de 40%: 107


exigência pelo menor score,1 fator,2 fatores,fora da matriz,inelegível (score abaixo de 500)
exigência,,,,
1 fator,290,0,4,0
2 fatores,0,274,8,0
fora da matriz,0,0,140,0
inelegível (score abaixo de 500),0,0,0,2


**Onde olhar:** as duas contagens impressas e, na tabela, os números fora da diagonal.

**O que se vê:** dos 142 fora da matriz, 107 têm score de 580 ou mais e PTI acima de 40%; só 35 saem por score abaixo de 580, 2 deles abaixo de 500. Os 142 juntam os 140 fora da matriz e esses 2 inelegíveis. Com o menor score entre os tomadores, 12 pedidos que estavam dentro da matriz passam para fora (4 que pediam um fator e 8 que pediam dois).

**O que significa:** a maior parte dos casos fora da matriz não depende da escala do score. Nenhuma linha da matriz aceita PTI acima de 40%, então os 107 só podem ter sido aprovados pelo sistema automático. Nos 35 de score baixo, a leitura depende da diferença entre VantageScore e MDCS, e não dá para afirmar a via. Trocar o score do respondente pelo menor entre os tomadores mexe em poucos casos (12 de 718), mas todos no sentido de deixar a norma mais rígida.

**Decisão: os 142 fora da matriz saem da nota**, como os de 51% a 60%. Cada caso fica marcado pelo limite que o tirou, "fora da tabela manual (PTI acima de 40%)" para os 107 e "score da NSMO abaixo de 580" para os 35, e os dois grupos são reportados à parte, com a contagem. Nenhum deles mede o raciocínio do agente. Os 107 só podem ter sido aprovados pelo sistema automático, e nos 35 a discordância viria da via automática ou da diferença de escala do score. O critério vem da norma e é fixado antes de rodar o agente, então não infla a nota. A remoção é aplicada na preparação dos dados. Se a prestação do PTI for incompleta (tabela da seção 2.5), há mais casos acima de 40% do que os 107, e o teste com limite de PTI mais apertado na avaliação mede esse efeito.

**Decisão: resposta condicional.** Nos 477 pedidos em que a exigência depende de um fator que a base não mostra, a resposta esperada do agente é suspender pedindo o fator. Contar uma aprovação com compensação como acerto premiaria a aposta, porque, como a NSMO só tem aprovados, um agente que aprovasse todos acertaria todos. Esses casos saem da concordância com o banco e são avaliados contra a norma aplicada em código, contando quantas vezes o agente suspendeu pedindo o fator certo, aprovou sem verificar ou negou. Negar conta como erro, porque o banco aprovou. A mesma regra vale para a compra com CLTV acima de 96,5% (seção 6).

**Decisão: os 99 de exigência verificada viram checagem, não comparação com as referências.** O mesmo argumento vale para eles. Todos têm DTI de 44% a 47% e foram aprovados, pelo banco e pela norma. Ali, a referência "aprovar todos" acerta 100% e a regra fixa de DTI (aprova até 43%) acerta 0%, só pela forma como o grupo foi montado, e nenhum agente fica acima das duas. Os 99 medem outra coisa, se o agente evita negar ou suspender quem a norma aprova. Negar ou suspender um deles conta como erro.

**Pergunta:** a regra dos 400 casos (seção 2) previa cerca de 200 pedidos na nota final. Depois dos cortes, qual é a margem de erro de cada grupo, comparada com a desses 200?

In [51]:
def margem(n: int, taxa: float = 0.8) -> float:
    """Meia largura do intervalo de 95% de uma taxa, em pontos percentuais."""
    return 1.96 * (taxa * (1 - taxa) / n) ** 0.5 * 100

verificaveis = (caso['leitura da norma'] == 'exigência verificada').sum()
condicionais = len(caso) - verificaveis - len(fora)
metade_verif = verificaveis // 2
metade_cond = condicionais // 2

print('Margem para uma taxa perto de 80%:')
print(f'  previsto na regra dos 400 (200 na nota): ± {margem(200):.1f} pontos')
print(f'  {condicionais} avaliados contra a norma, metade na nota ({metade_cond}): '
      f'± {margem(metade_cond):.1f} pontos')
print(f'  {verificaveis} com exigência verificada, metade na nota ({metade_verif}): '
      f'± {margem(metade_verif):.1f} pontos')
print(f'  {verificaveis} com exigência verificada, todos na nota: '
      f'± {margem(verificaveis):.1f} pontos')

Margem para uma taxa perto de 80%:
  previsto na regra dos 400 (200 na nota): ± 5.5 pontos
  477 avaliados contra a norma, metade na nota (238): ± 5.1 pontos
  99 com exigência verificada, metade na nota (49): ± 11.2 pontos
  99 com exigência verificada, todos na nota: ± 7.9 pontos


**Onde olhar:** as três últimas linhas, comparadas com a primeira.

**O que se vê:** os 477 avaliados contra a norma ficam com ±5,1 pontos mesmo divididos ao meio, perto do previsto (±5,5). Os 99 com exigência verificada ficam com ±11,2 se forem divididos ao meio e com ±7,9 se forem todos para a nota final.

**O que significa:** a regra dos 400 continua valendo para os casos avaliados contra a norma, que são o teste principal da NSMO. A checagem dos 99 é complementar, com margem larga.

**Decisão: os 99 vão inteiros para a nota final.** O ajuste do agente usa casos do HMDA e a parte de ajuste dos 477, então os 99 não precisam ser divididos, e a margem da checagem fica menor. A separação é feita na preparação dos dados.

**Pergunta:** a resposta condicional também vale para a compra com CLTV acima de 96,5% (decisão acima). Quantas compras do HMDA ela tira da concordância com o banco?

In [52]:
# Compras do HMDA com CLTV acima do teto de compra, por decisão
compras_hmda = hmda[hmda['loan_purpose'] == '1']
cltv_hmda = pd.to_numeric(compras_hmda['loan_to_value_ratio'], errors='coerce')
acima_teto = compras_hmda[cltv_hmda > CLTV_CAP['1']]
print('compras com CLTV acima de 96,5%:', len(acima_teto))
acima_teto['action_taken'].map(ACTION_TAKEN).value_counts().to_frame('pedidos')

compras com CLTV acima de 96,5%: 168979


,pedidos
action_taken,
originado,136485
negado,20738
"pré aprovação aprovada, não aceita",4230
"aprovado, não aceito",4112
pré aprovação negada,3414


**Onde olhar:** o total e as duas primeiras linhas.

**O que se vê:** 168.979 compras do HMDA passam de 96,5%, entre elas 136.485 originadas e 20.738 negadas.

**O que significa:** essas compras saem da concordância com o banco e são avaliadas contra a norma aplicada em código, como os 477 da NSMO. Uma compra pode estar também na faixa de DTI de 50% a 60%, então esta contagem e a da seção 5.2 não se somam.

## 8. Síntese

**Checagens.** Na Checagem 1, a regra dos 25% passou e a regra do dobro não, porque o DTI do HMDA separa negados de aprovados só nas pontas. Na Checagem 2, as duas regras passaram, com DTI e score exatos e mais de 400 casos com score e reserva de 44% a 60%. Depois dos cortes das seções 5.2 e 7.2, esse número basta para os 477 avaliados contra a norma. Os 99 casos de exigência verificada servem só de checagem, com margem larga (seção 7.2). O projeto segue com a medição por faixa.

**Decisões tomadas neste notebook:**

1. Métrica do HMDA por faixa de DTI, com o papel de cada faixa definido na seção 1.7 e a faixa do meio dividida em 50% na seção 5.2.
2. Os negados por garantia ficam na avaliação, com o resultado reportado à parte por finalidade (seção 1.5).
3. Pedidos fechados por incompletude saem da comparação com o banco (seção 4.1); a remoção é aplicada na preparação dos dados.
4. A faixa do meio (44% a 60%) é dividida no teto manual. De 44% a 50% é avaliada, e de 51% a 60% vai para relatório à parte (seção 5.2).
5. O índice da norma usa a Update 18 do Handbook (seção 5.4).
6. Premissa: numa compra, CLTV acima de 96,5% vira condição a confirmar (seguro financiado ou segundo financiamento), e não motivo de negativa (seção 6).
7. Na NSMO, os 142 casos de 44% a 50% fora da matriz (107 por PTI acima de 40% e 35 por score abaixo de 580) saem da nota e vão para relatório à parte (seção 7.2).
8. Quando a norma só permite aprovar mediante uma comprovação que o analista pediria ao cliente e a base não a mostra (fator compensatório na NSMO, seguro financiado ou segundo financiamento na compra com CLTV acima de 96,5%), a resposta esperada é suspender pedindo a comprovação. Esses casos são avaliados contra a norma aplicada em código, à parte da concordância com o banco (seção 7.2).
9. O PTI ausente no HMDA é lacuna da base, porque o banco calcula esse número sozinho. Até 43%, a resposta esperada continua sendo aprovar, com o PTI marcado como não informado, e a concordância sai em duas partes, até 35% e de 36% a 43% (seção 7.1).
10. A finalidade do pedido entra no perfil do caso, porque no refinanciamento com saque o teto de CLTV é outro, e o motivo da negativa fica visível para o agente (seções 1.5 e 6). O sorteio dos grupos de avaliação mantém a proporção de cada finalidade, que muda muito a chance de negativa (seção 6).
11. Os 99 pedidos da NSMO de exigência verificada não comparam o agente com as referências, porque ali "aprovar todos" acerta tudo e a regra fixa de DTI erra tudo. Eles checam se o agente evita negar ou suspender quem a norma aprova e vão inteiros para a nota final, para a margem cair de cerca de 11 para cerca de 8 pontos (seção 7.2).

**Achados que seguem para as próximas fases:**

- O agente precisa conhecer o Streamline, que explica o refinanciamento sem saque aprovado com CLTV acima de 100% e o CLTV vazio em 41,6% desses refinanciamentos (seções 1.5 e 1.6).
- DTI acima de 60% aparece em negativas por outros motivos, o que afeta a avaliação do motivo da negativa (seção 1.6).
- Os campos da NSMO medem score, reserva, PTI e DTI de forma diferente da norma, e a tabela do fim da seção 2.5 diz o que se faz com cada diferença. A matriz e a reserva mínima não mudaram de 2016 a 2026 (seção 5.4).